# Arabic document VLM, part 2 (v5): four-stage training with the data and training changes from the v4 analysis

**Runtime: GPU, L4** (*Runtime > Change runtime type > L4*). A T4 works for short runs but is about twice as slow and
cannot run the optional decoder pretraining comfortably.

**Before this notebook:** run part 1 (assembles the model, unchanged from v4) and **part 1b v5** (downloads and
prepares all the data into `MyDrive/arabic_vlm/data_v5`, or `arabic_vlm_data_v5.zip` uploaded to `MyDrive/arabic_vlm/`).
This notebook downloads nothing for training: it copies the prepared data to the Colab disk and reads it from there.

**What v5 changes, and why.** The v4 logs showed three things: Arabic handwriting training came from only 121 KHATT
lines (the benchmark-overlap filter removed the rest), the speed benchmark cut long diacritised lines at 64 tokens,
and the 0.5B decoder's Arabic prior produced plausible-but-wrong words on printed classical text. Every change below
is a switch in section A, so a run can be as small as "v4 plus the new data" or as large as "pretrain the decoder and
train every stage again".

| Area | v4 | v5 | Switch |
| --- | --- | --- | --- |
| Arabic handwriting data | 121 KHATT lines | KHATT with a relaxed filter (thousands of lines, overlap flagged) + Muharaf (about 22,000 historical lines) | `KHATT_STRICT` |
| Leakage | assumed small | measured: benchmark handwriting lines scored separately when they share 4 words in a row with training lines | automatic |
| English data | 8 fonts, 14% lines + 8% IAM | 40 fonts, 10% lines + 10% IAM + SynthDoG pages + bilingual lines | `USE_SYNTHDOG` |
| Forms | 6 fixed fields, one layout | 4–8 of 12 fields, four layouts, stamps, boxes, handwriting-like values, perspective | automatic |
| Token limit at inference | fixed 64 / 128 / 160 | from the image size, with per-task ceilings | `max_tokens_for` |
| Image resolution | up to 640 visual tokens | up to 1,024 in training, 1,280 at inference, plus an ablation | `MAX_PIXELS` |
| Vision encoder | LoRA on all 32 blocks | LoRA on the first 24 blocks, the last 8 trained in full at a lower learning rate | `VISION_FULL_BLOCKS` |
| Decoder | as shipped | optional continued pretraining on Arabic text before stage 1 (then stages 1 and 2 train again) | `PRETRAIN_DECODER` |
| Stage 4 | lr 2e-4 | lr 1e-4, and the stage-3 checkpoint is also scored on the public sets | automatic |
| Scoring | raw CER/WER, 200 images | raw, without diacritics and Arabic-normalised CER/WER, 95% bootstrap intervals, up to 500 images per set, every prediction saved | `PUBLIC_N` |
| Decoding | greedy | greedy, plus an ablation of repetition control and beam search on the handwriting sets | `DECODING` |
| Real documents | none | scored if `real_docs` was prepared in part 1b | automatic |

**Time on an L4 (full run size):** stage 3 about 12–14 hours, stage 4 about 4 hours, public scoring about 1.5 hours.
With `PRETRAIN_DECODER = True` add about 2 hours for the pretraining and 6 hours for stages 1 and 2. Every stage saves
its progress to Drive at each check, so it can run over several sessions: if Colab stops, rerun sections A–F and the
stage's cells.

**Nothing in this notebook has produced results yet.** The v4 numbers quoted in the markdown are from the v4 run;
v5 numbers appear only when you run it.

## A. Install, imports and settings

In [ ]:
%pip install -q arabic-reshaper python-bidi

In [ ]:
import os, sys, re, gc, io, json, math, time, random, shutil, subprocess, importlib, unicodedata, zipfile
import numpy as np
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

# Colab ships an old torchao that makes PEFT (the LoRA library) fail to import. This notebook does not use torchao.
try:
    from peft.import_utils import is_torchao_available
    is_torchao_available()
except ImportError as e:
    if "torchao" in str(e):
        print("Removing an incompatible optional package:", e)
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
        importlib.invalidate_caches()
from peft import LoraConfig, get_peft_model

QUICK_TEST = os.environ.get("ARVL_QUICK_TEST", "0") == "1"   # tiny CPU run, used only to test this notebook

# --- which run this is ---------------------------------------------------------------------
RUN_NAME = "v5"                         # names this run's checkpoints, progress files and results
PREVIOUS_RUN = "v4"                     # its final checkpoint is scored beside v5 on the public sets, if it is still on Drive

# --- the v5 switches ------------------------------------------------------------------------
PRETRAIN_DECODER = False                # True: stage 0b pretrains the 0.5B decoder on Arabic text, then stages 1 and 2 train again (adds ~8 h)
DECODER_PRETRAIN_TOKENS = 30_000_000    # tokens seen in stage 0b (about 2 h on an L4)
VISION_FULL_BLOCKS = 8                  # last N vision blocks trained in full in stages 3 and 4 (0 = LoRA on every block, as v4)
LR_FULL = 2e-5                          # learning rate for those blocks (LoRA adapters and the projector use the stage's lr)
KHATT_STRICT = False                    # True: train only on KHATT lines that share no 4 words in a row with KITAB's handwriting lines (v4 behaviour)
USE_SYNTHDOG = True                     # rendered English document pages, if part 1b saved them
MIN_PIXELS, MAX_PIXELS = 32 * 28 * 28, 1024 * 28 * 28   # image size limits in training and evaluation (v4: 640 visual tokens)
MAX_PIXELS_INFER = 1280 * 28 * 28       # the larger limit tried in the resolution ablation (and for your own documents)
DECODING = {}                           # extra generation settings for every evaluation, e.g. {"no_repeat_ngram_size": 4}; the ablation tries them
OUTPUT_CHAR_MAP = str.maketrans({"ی": "ي", "ک": "ك"})   # Farsi yeh and kaf -> Arabic yeh and kaf, applied to every model's output
STAGE4_LR = 1e-4                        # v4 used 2e-4 and lost a little reading accuracy in stage 4
PUBLIC_N = 500 if not QUICK_TEST else 4 # images per public set (None = whole subsets; KITAB's line subsets have 200-800 images)

# --- how much to train ----------------------------------------------------------------------
RUN_SIZE = "full"                       # "short" = a quick end-to-end check, "full" = the real run
STAGE_EXAMPLES = {
    "short": {1: 3_200, 2: 3_200, 3: 6_400, 4: 3_200},
    "full":  {1: 30_000, 2: 25_000, 3: 90_000, 4: 16_000},
}[RUN_SIZE]
if RUN_SIZE == "short":
    DECODER_PRETRAIN_TOKENS = 400_000
CHECKS_PER_STAGE = 12                   # progress checks (loss, CER, WER) during each stage
DECODER_LORA_IN_STAGE3 = True           # False = Grab's recipe exactly (decoder frozen until stage 4)
WARM_START_PROJECTOR = True             # stage 1 only: start the projector from Qwen's trained one where shapes allow
REUSE_STAGES_1_AND_2 = (os.environ.get("ARVL_REUSE", "1") == "1") and not PRETRAIN_DECODER   # use the v3 stage 1 + 2 checkpoints
BASE_VL_ID = os.environ.get("ARVL_VL_ID", "Qwen/Qwen2.5-VL-3B-Instruct")   # where the vision encoder came from
BASELINE_ID = os.environ.get("ARVL_BASELINE", "Qwen/Qwen2.5-VL-3B-Instruct")  # the model we compare against

# --- where checkpoints live ---------------------------------------------------------------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
BASE_DIR = os.environ.get("ARVL_BASE", "/content/drive/MyDrive/arabic_vlm" if IN_COLAB else "arabic_vlm")
early = "" if REUSE_STAGES_1_AND_2 else f"_{RUN_NAME}"
CHECKPOINTS = {
    0: os.environ.get("ARVL_ASSEMBLED", os.path.join(BASE_DIR, "stage0_assembled")),   # from part 1
    "0b": os.path.join(BASE_DIR, f"stage0b_decoder_{RUN_NAME}"),                        # optional decoder pretraining
    1: os.path.join(BASE_DIR, f"stage1{early}"),
    2: os.path.join(BASE_DIR, f"stage2{early}"),
    3: os.path.join(BASE_DIR, f"stage3_{RUN_NAME}"),
    4: os.path.join(BASE_DIR, f"stage4_{RUN_NAME}"),
}
PREVIOUS_FINAL = os.path.join(BASE_DIR, f"stage4_{PREVIOUS_RUN}")   # the v4 model, compared in the results if it is still there
KEEP_ALL_CHECKPOINTS = True   # each checkpoint is ~4.6 GB; False deletes the previous stage's checkpoint after saving
os.makedirs(BASE_DIR, exist_ok=True)

# --- the prepared data (from part 1b v5) ---------------------------------------------------
DATA_ON_DRIVE = os.environ.get("ARVL_DATA", os.path.join(BASE_DIR, "data_v5"))
DATA_ZIP = os.path.join(BASE_DIR, "arabic_vlm_data_v5.zip")
DATA_DIR = "/content/arabic_vlm_data_v5" if IN_COLAB else os.environ.get("ARVL_DATA_LOCAL", DATA_ON_DRIVE)
if not os.path.isdir(DATA_DIR):
    start = time.time()
    if os.path.isdir(DATA_ON_DRIVE):
        shutil.copytree(DATA_ON_DRIVE, DATA_DIR)                     # Drive -> local disk: much faster to read
    elif os.path.exists(DATA_ZIP):
        zipfile.ZipFile(DATA_ZIP).extractall(os.path.dirname(DATA_DIR))
    else:
        raise FileNotFoundError(f"No prepared data: run part 1b v5, which saves {DATA_ON_DRIVE} (or upload {DATA_ZIP}).")
    print(f"data copied to {DATA_DIR} in {(time.time() - start) / 60:.1f} min")
MANIFEST = json.load(open(os.path.join(DATA_DIR, "manifest.json")))
assert MANIFEST.get("version") == "v5", f"this notebook needs the v5 data folder from part 1b v5, found {MANIFEST.get('version')}"
print(f"data: version {MANIFEST['version']}, prepared {MANIFEST['created']}")

# --- hardware -----------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if DEVICE.type == "cuda" else 0
FULL_FINETUNE = GPU_GB >= 39            # A100 / H100: train weights directly. T4 / L4: LoRA (+ full vision blocks) in stages 2-4
USE_AMP = DEVICE.type == "cuda"         # mixed precision. Set False if training keeps reporting non-finite losses
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported(including_emulation=False)) else torch.float16
LORA_R, LORA_ALPHA = 32, 64

# --- evaluation set sizes -----------------------------------------------------------------
N_EVAL = 3 if QUICK_TEST else 48        # images per evaluation set at the start and end of each stage
N_CHECK = 2 if QUICK_TEST else 12       # images per set in the quick checks during training
if QUICK_TEST:
    STAGE_EXAMPLES = {s: int(os.environ.get("ARVL_EXAMPLES", "8")) for s in STAGE_EXAMPLES}
    DECODER_PRETRAIN_TOKENS = 4_000
    CHECKS_PER_STAGE = 2
    VISION_FULL_BLOCKS = min(VISION_FULL_BLOCKS, 1)

print(f"run {RUN_NAME}: checkpoints in {BASE_DIR} -> " + ", ".join(os.path.basename(p) for p in CHECKPOINTS.values()))
print(f"run size: {RUN_SIZE} -> examples per stage {STAGE_EXAMPLES}"
      + (f", decoder pretraining on {DECODER_PRETRAIN_TOKENS:,} tokens" if PRETRAIN_DECODER else ""))
print(f"GPU: {GPU_GB:.0f} GB -> {'full fine-tuning' if FULL_FINETUNE else 'LoRA'} in stages 2-4 | "
      f"mixed precision: {AMP_DTYPE if USE_AMP else 'off'} | vision blocks trained in full: {VISION_FULL_BLOCKS}")
assert os.path.isdir(CHECKPOINTS[0]), (
    f"Cannot find {CHECKPOINTS[0]}. Run the part-1 notebook, which saves the assembled model there.")

## B. Drawing text
Arabic letters change shape depending on their position in a word, and lines read right to left. Pillow handles
both, and places diacritics correctly, when it has the "raqm" text-layout library (Colab does).

- `render_lines(lines, rng, style)` draws one or more lines in one of four styles: **page** (paper, dark ink),
  **colour** (pastel background, coloured ink), **slide** (light text on a dark or saturated background, sometimes
  centred) and **sign** (a bold font on a strong colour, for single words). Font, size (20–46 px), weight and
  alignment vary. Lines up to 20 words make images up to 2,200 px wide.
- `damage(img, level)` makes an image look like a scan or phone photo: tilt, blur, noise, JPEG artefacts,
  low resolution and, new in v5, a perspective skew at level 2. Level 0 = clean, 1 = light, 2 = heavy.

In [ ]:
from PIL import Image, ImageDraw, ImageFont, ImageFilter, features

FONT_DIR = os.path.join(DATA_DIR, "fonts")
FONT_LIST = json.load(open(os.path.join(FONT_DIR, "fonts.json")))
font_path = lambda name: os.path.join(FONT_DIR, name)
SPECIAL = set(FONT_LIST["handwriting_like"]) | set(FONT_LIST["display"]) | set(FONT_LIST.get("mono", []))
ARABIC_BODY_FONTS = [font_path(f) for f in FONT_LIST["arabic"] if f not in SPECIAL]
ARABIC_SPECIAL_FONTS = [font_path(f) for f in FONT_LIST["arabic"] if f in SPECIAL]
ARABIC_FONTS = ARABIC_BODY_FONTS + ARABIC_SPECIAL_FONTS
ENGLISH_BODY_FONTS = [font_path(f) for f in FONT_LIST["english"] if f not in SPECIAL]
ENGLISH_SPECIAL_FONTS = [font_path(f) for f in FONT_LIST["english"] if f in SPECIAL]
ENGLISH_FONTS = ENGLISH_BODY_FONTS + ENGLISH_SPECIAL_FONTS
DISPLAY_FONTS = {"ar": [font_path(f) for f in FONT_LIST["display"] if f in FONT_LIST["arabic"]] or ARABIC_BODY_FONTS,
                 "en": [font_path(f) for f in FONT_LIST["display"] if f in FONT_LIST["english"]] or ENGLISH_BODY_FONTS}
MONO_FONTS = [font_path(f) for f in FONT_LIST.get("mono", [])] or ENGLISH_BODY_FONTS
HAND_FONTS = {"ar": [font_path(f) for f in FONT_LIST["handwriting_like"] if f in FONT_LIST["arabic"]] or ARABIC_BODY_FONTS,
              "en": [font_path(f) for f in FONT_LIST["handwriting_like"] if f in FONT_LIST["english"]] or ENGLISH_BODY_FONTS}
HAS_RAQM = features.check("raqm")
print(f"fonts: {len(ARABIC_BODY_FONTS)} Arabic body, {len(ARABIC_SPECIAL_FONTS)} Arabic handwriting-like or display, "
      f"{len(ENGLISH_BODY_FONTS)} English body, {len(ENGLISH_SPECIAL_FONTS)} English handwriting-like, display or mono | raqm text layout: {HAS_RAQM}")
_fonts = {}


def get_font(path, size):
    if (path, size) not in _fonts:
        engine = ImageFont.Layout.RAQM if HAS_RAQM else ImageFont.Layout.BASIC
        _fonts[(path, size)] = ImageFont.truetype(path, size, layout_engine=engine)
    return _fonts[(path, size)]


def is_arabic(text):
    """True when the text has Arabic letters. Arabic-Indic digits alone (a phone number, a date) do not count: on
    their own they are written left to right."""
    return re.search(r"[؀-ٟ٪-ۿ]", text) is not None


def prepare(text):
    """Return (text to draw, extra drawing options) so Arabic comes out joined and right-to-left."""
    if HAS_RAQM:
        return text, ({"direction": "rtl", "language": "ar"} if is_arabic(text) else {"direction": "ltr"})
    if is_arabic(text):
        import arabic_reshaper
        from bidi.algorithm import get_display
        return get_display(arabic_reshaper.reshape(text)), {}
    return text, {}


def draw_text(draw, x_left, x_right, y, text, font, color, align, stroke=0):
    shown, options = prepare(text)
    width = draw.textlength(shown, font=font, **options)
    x = {"right": x_right - width, "left": x_left, "center": (x_left + x_right - width) / 2}[align]
    draw.text((x, y), shown, font=font, fill=color, stroke_width=stroke, stroke_fill=color, **options)


def typeset_ltr(text):
    """Keep Arabic-Indic digit groups in reading order when drawn. Any bidi layout engine displays the groups of
    "٠٥٠ ١٢٣ ٤٥٦٧" in reverse order (the spaces between Arabic numbers resolve to right-to-left), which is what a naively
    typed number looks like on a real form; a left-to-right mark before each separator is what a typesetter adds to keep
    the groups in order. The marks are invisible and never part of the answer text."""
    if re.search(r"[٠-٩]", text) and not is_arabic(text):
        return re.sub(r"(?<=[٠-٩])(?=[\s\-/.:])", "‎", text)
    return text


def pick_font(rng, arabic, style):
    if style == "sign" and rng.random() < 0.6:
        return rng.choice(DISPLAY_FONTS["ar" if arabic else "en"])
    if not arabic:
        return rng.choice(ENGLISH_SPECIAL_FONTS if rng.random() < 0.15 else ENGLISH_BODY_FONTS)
    return rng.choice(ARABIC_SPECIAL_FONTS if rng.random() < 0.15 else ARABIC_BODY_FONTS)


def colours(rng, style):
    """(background, ink) for a style."""
    if style == "page":
        return tuple(rng.randint(232, 255) for _ in range(3)), tuple([rng.randint(0, 60)] * 3)
    if style == "colour":
        return tuple(rng.randint(200, 255) for _ in range(3)), tuple(rng.randint(0, 110) for _ in range(3))
    if style == "slide" and rng.random() < 0.5:
        return tuple(rng.randint(225, 255) for _ in range(3)), rng.choice([(20, 40, 120), (150, 20, 20), (20, 90, 40), (30, 30, 30)])
    return tuple(rng.randint(0, 140) for _ in range(3)), tuple(rng.randint(225, 255) for _ in range(3))   # slide / sign


def background(rng, size, colour):
    """A flat colour, or a soft gradient (slides and signs)."""
    if rng.random() < 0.7:
        return Image.new("RGB", size, colour)
    w, h = size
    other = np.array([min(255, max(0, c + rng.randint(-50, 50))) for c in colour], dtype="float32")
    t = np.linspace(0, 1, w if rng.random() < 0.5 else h, dtype="float32")
    ramp = np.array(colour, dtype="float32")[None] * (1 - t[:, None]) + other[None] * t[:, None]
    grid = np.broadcast_to(ramp[None], (h, w, 3)) if len(t) == w else np.broadcast_to(ramp[:, None], (h, w, 3))
    return Image.fromarray(grid.astype("uint8"))


def render_lines(lines, rng, style=None, min_size=20):
    style = style or rng.choices(["page", "colour", "slide", "sign"], [0.6, 0.2, 0.15, 0.05])[0]
    arabic = any(is_arabic(l) for l in lines)
    path = pick_font(rng, arabic, style)
    size = rng.randint(30, 56) if style == "sign" else rng.randint(min_size, max(min_size, 46))
    probe = ImageDraw.Draw(Image.new("RGB", (10, 10)))
    widest = max(probe.textlength(prepare(l)[0], font=get_font(path, size), **prepare(l)[1]) for l in lines)
    if widest > 2100:                                          # keep very long lines under about 2,200 px
        size = max(min(min_size, 18), int(size * 2100 / widest))
        widest = max(probe.textlength(prepare(l)[0], font=get_font(path, size), **prepare(l)[1]) for l in lines)
    font = get_font(path, size)
    pad, line_height = rng.randint(10, 32), int(size * rng.uniform(1.45, 1.9))
    width = int(widest + 2 * pad + rng.randint(0, 80))
    bg, ink = colours(rng, style)
    img = background(rng, (width, 2 * pad + line_height * len(lines)), bg)
    draw = ImageDraw.Draw(img)
    stroke = 1 if rng.random() < (0.4 if style in ("slide", "sign") else 0.1) else 0     # bold
    centred = style in ("slide", "sign") and rng.random() < 0.4
    if style == "page" and rng.random() < 0.08:                # a table rule or underline now and then
        y = pad + line_height * len(lines) - int(size * 0.25)
        draw.line([(pad // 2, y), (width - pad // 2, y)], fill=tuple(min(255, c + 90) for c in ink), width=1)
    for i, line in enumerate(lines):
        align = "center" if centred else ("right" if is_arabic(line) else "left")
        draw_text(draw, pad, width - pad, pad + i * line_height, line, font, ink, align, stroke)
    return img


def perspective_coeffs(target, source):
    """Coefficients for Image.transform(PERSPECTIVE): the corners `source` of the input land on `target` in the output."""
    rows = []
    for (x, y), (u, v) in zip(target, source):
        rows.append([x, y, 1, 0, 0, 0, -u * x, -u * y])
        rows.append([0, 0, 0, x, y, 1, -v * x, -v * y])
    return np.linalg.solve(np.array(rows, dtype="float64"), np.array(source, dtype="float64").reshape(8)).tolist()


def perspective(img, rng, strength):
    """A mild perspective skew, like a page photographed at an angle."""
    w, h = img.size
    dx, dy = w * strength, h * strength
    target = [(rng.uniform(0, dx), rng.uniform(0, dy)), (w - rng.uniform(0, dx), rng.uniform(0, dy)),
              (w - rng.uniform(0, dx), h - rng.uniform(0, dy)), (rng.uniform(0, dx), h - rng.uniform(0, dy))]
    return img.transform((w, h), Image.PERSPECTIVE, perspective_coeffs(target, [(0, 0), (w, 0), (w, h), (0, h)]),
                         Image.BILINEAR, fillcolor=img.getpixel((0, 0)))


def damage(img, rng, level):
    if level == 0:
        return img
    if level >= 2 and rng.random() < 0.3:                                      # photographed at an angle (v5)
        img = perspective(img, rng, 0.04)
    if rng.random() < 0.5 * level:                                            # slight tilt
        img = img.rotate(rng.uniform(-1.5, 1.5) * level, expand=True, fillcolor=img.getpixel((0, 0)))
    if rng.random() < 0.5 * level:                                            # blur
        img = img.filter(ImageFilter.GaussianBlur(rng.uniform(0.3, 0.9) * level))
    if rng.random() < 0.4 * level:                                            # sensor noise
        a = np.asarray(img).astype("float32")
        a += np.random.default_rng(rng.randint(0, 10**9)).normal(0, 6 * level, a.shape)
        img = Image.fromarray(a.clip(0, 255).astype("uint8"))
    if rng.random() < 0.4 * level:                                            # JPEG compression
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=rng.randint(30, 70))
        img = Image.open(io.BytesIO(buf.getvalue())).convert("RGB")
    if rng.random() < 0.3 * level:                                            # low-resolution capture
        w, h = img.size
        s = rng.uniform(0.6, 0.85)
        img = img.resize((max(28, int(w * s)), max(28, int(h * s)))).resize((w, h))
    return img

## C. Loading the prepared data

Everything comes from the folder part 1b v5 made: sentences to draw (Wikipedia Arabic and English, diacritised
Arabic), real handwriting (KHATT and Muharaf Arabic, IAM English), SynthDoG English pages, the KITAB-Bench
evaluation sets and, if you added them, your own documents. Part 1b already removed anything that overlaps the
benchmark and flagged KHATT and Muharaf lines that share 4 words in a row with a benchmark line.

The sentences are split once into a **training pool** and a **held-out pool**, so the drawn evaluation images
never contain text the model trained on. KHATT, Muharaf and IAM are evaluated on their own test splits.

In [ ]:
from datasets import load_from_disk

DIACRITICS = re.compile(r"[ً-ْٰـ]")   # tashkeel, dagger alif, tatweel


def strip_marks(text):
    return DIACRITICS.sub("", text)


def read_text(name):
    return json.load(open(os.path.join(DATA_DIR, "text", name), encoding="utf-8"))


def split_pool(sentences):
    sentences = list(sentences)
    random.Random(0).shuffle(sentences)                 # same split every time the notebook runs
    cut = max(1, len(sentences) // 20)
    return sentences[cut:] or sentences, sentences[:cut]   # (training pool, held-out pool)


def read_images_set(name):
    path = os.path.join(DATA_DIR, "datasets", name)
    return load_from_disk(path) if os.path.isdir(path) else None


data_start = time.time()
POOLS = {}
POOLS["msa"] = split_pool(read_text("wikipedia_ar.json"))
POOLS["tashkeel"] = split_pool(read_text("tashkeel_ar.json"))
POOLS["classical"] = POOLS["tashkeel"]                  # the same sentences, drawn with their diacritics removed
POOLS["en"] = split_pool(read_text("wikipedia_en.json"))
IAM = {split: read_images_set(f"iam_{split}") for split in ["train", "test"]}
KHATT = {split: read_images_set(f"khatt_{split}") for split in ["train", "test"]}
MUHARAF = {split: read_images_set(f"muharaf_{split}") for split in ["train", "validation", "test"]}
if KHATT_STRICT and "shares_4gram" in KHATT["train"].column_names:
    KHATT["train"] = KHATT["train"].filter(lambda r: not r["shares_4gram"])
SYNTHDOG = read_images_set("synthdog_en") if USE_SYNTHDOG else None
REAL_DOCS = read_images_set("real_docs")
for name in ["msa", "tashkeel", "en"]:
    print(f"{name:9s} sentences: {len(POOLS[name][0]):,} train / {len(POOLS[name][1]):,} held out")
flagged = sum(KHATT["train"]["shares_4gram"]) if "shares_4gram" in KHATT["train"].column_names else 0
print(f"handwriting: KHATT {len(KHATT['train']):,} train ({flagged:,} share 4 words with the benchmark) / {len(KHATT['test']):,} test, "
      f"Muharaf {len(MUHARAF['train']):,} train / {len(MUHARAF['test']):,} test, IAM {len(IAM['train']):,} train / {len(IAM['test']):,} test")
print(f"SynthDoG pages: {len(SYNTHDOG) if SYNTHDOG is not None else 0:,} | your documents: {len(REAL_DOCS) if REAL_DOCS is not None else 0}")
assert len(KHATT["train"]) > 0 and len(MUHARAF["train"]) > 0, "no handwriting training lines: check part 1b v5"
print(f"data ready in {(time.time() - data_start) / 60:.1f} min")

## D. Training examples
Every training example is a triple **(image, instruction, answer)**:

| Function | Image | Answer |
| --- | --- | --- |
| `text_line_example` | 1 line of 1–20 words, or a block of 2–5 lines, drawn from a text source (`msa`, `tashkeel`, `classical`, `en`) in one of the four styles | the exact text |
| `bilingual_line_example` (v5) | an Arabic label with an English or numeric value, or an Arabic sentence with an English term inside | the exact text |
| `scene_word_example` | 1–3 words in a sign style, tilted and damaged, like a crop from a street photo | the words |
| `khatt_example`, `muharaf_example` | a real handwritten Arabic line (KHATT: modern, Muharaf: historical manuscripts) | its transcription |
| `iam_example` | a real handwritten English line from IAM | its transcription |
| `synthdog_example` (v5) | a rendered English page on a paper photo (SynthDoG) | its text |
| `form_example` (v5) | a bilingual form with 4–8 of 12 fields in one of four layouts, with stamps, boxes and handwriting-like values | the fields as JSON |

**Digits and direction.** Arabic-Indic digit groups separated by spaces or hyphens ("٠٥٠ ١٢٣ ٤٥٦٧") are displayed in
reverse group order by every bidi layout engine, including the one that draws these images, because the spaces between
Arabic numbers resolve to right-to-left; typesetters avoid this with a left-to-right mark before each separator. Real
forms show both. The forms and bilingual lines therefore draw such values both ways (half and half), and the answer is
always the logical text (or, for forms, the normalised value), never the visual order. Emirates ID numbers are drawn in
Western digits only, as on the card, and phone numbers mostly so.

Two instructions are used: the usual one asks for the text exactly as written; the *plain* one, used for the
classical-style lines (drawn without diacritics) and for scene words, asks for Arabic without diacritics. That gives
a product a switch for "no marks", and it teaches the model not to invent marks that are not in the image. Every
evaluation in this notebook uses the usual instruction, for every model.

Real handwriting is lightly augmented (tilt, blur, noise) because the model sees each line several times.
Each stage mixes these in its own proportions (shown in the stage sections). Examples are generated from a seed,
so a resumed run continues with exactly the examples it would have seen.

In [ ]:
TO_ARABIC_DIGITS = str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")
TO_WESTERN_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
OCR_INSTRUCTION = "Read all the text in the image exactly as written. اقرأ النص كما هو مكتوب."
OCR_PLAIN_INSTRUCTION = "Read all the text in the image. Write Arabic without diacritics. اقرأ النص بدون تشكيل."


def words_in_line(rng, block=False):
    if block:
        return rng.randint(3, 10)
    x = rng.random()
    if x < 0.08:
        return rng.randint(1, 2)
    if x < 0.45:
        return rng.randint(3, 8)
    if x < 0.80:
        return rng.randint(9, 14)
    return rng.randint(15, 20)


def text_piece(rng, pool, n):
    """n consecutive words, continuing into further sentences when one is too short."""
    words = rng.choice(pool).split()
    while len(words) < n:
        words += rng.choice(pool).split()
    start = rng.randint(0, len(words) - n)
    return " ".join(words[start:start + n])


LATIN_EXTRAS = ["PDF", "Invoice INV-2026", "ID", "Email", "UAE", "AED 250", "Ref A-17", "www.gov.ae", "VAT 5%", "Dubai Mall",
                "ERP", "Microsoft Teams", "Etihad", "ADNOC", "Block B-12", "iPhone 15", "Wi-Fi", "QR", "GPS", "COVID-19"]


def make_line(rng, source, heldout, n, extras):
    text = text_piece(rng, POOLS[source][1 if heldout else 0], n)
    if source == "classical":
        text = strip_marks(text)
    if source != "en" and extras and rng.random() < 0.25:     # a number, in Arabic-Indic or Western digits
        number = str(rng.choice([rng.randint(1, 99), rng.randint(100, 99999), rng.randint(1900, 2030)]))
        text += " " + (number.translate(TO_ARABIC_DIGITS) if rng.random() < 0.6 else number)
    if source != "en" and extras and rng.random() < 0.1:      # a Latin word inside the Arabic line
        text += " " + rng.choice(LATIN_EXTRAS)
    return text


def text_line_example(rng, language, damage_level, max_lines=1, heldout=False, source=None, style=None):
    source = "en" if language == "en" else (source or "msa")
    count = rng.randint(2, max_lines) if max_lines > 1 else 1
    lines = [make_line(rng, source, heldout, words_in_line(rng, block=count > 1), extras=damage_level > 0)
             for _ in range(count)]
    image = damage(render_lines(lines, rng, style, min_size=28 if damage_level == 2 else 20), rng, damage_level)
    instruction = OCR_PLAIN_INSTRUCTION if source == "classical" else OCR_INSTRUCTION
    return image, instruction, "\n".join(lines)


def scene_word_example(rng, heldout=False):
    language = "en" if rng.random() < 0.15 else "ar"
    source = "en" if language == "en" else rng.choice(["msa", "msa", "classical"])
    text = make_line(rng, source, heldout, rng.randint(1, 3), extras=False)
    img = render_lines([text], rng, style=rng.choice(["sign", "sign", "colour", "slide"]))
    img = img.rotate(rng.uniform(-4, 4), expand=True, fillcolor=img.getpixel((0, 0)))
    return damage(img, rng, rng.choice([1, 2])), OCR_PLAIN_INSTRUCTION if language == "ar" else OCR_INSTRUCTION, text


def handwriting_from(ds, rng, augment=True):
    row = ds[rng.randrange(len(ds))]
    img = row["image"].convert("RGB")
    return (damage(img, rng, rng.choice([0, 0, 1])) if augment else img), OCR_INSTRUCTION, row["text"]


def khatt_example(rng, split="train"):
    return handwriting_from(KHATT[split], rng, augment=split == "train")


def muharaf_example(rng, split="train"):
    return handwriting_from(MUHARAF[split], rng, augment=split == "train")


def iam_example(rng, split="train"):
    return handwriting_from(IAM[split], rng, augment=split == "train")


def synthdog_example(rng):
    row = SYNTHDOG[rng.randrange(len(SYNTHDOG))]
    return damage(row["image"].convert("RGB"), rng, rng.choice([0, 1])), OCR_INSTRUCTION, row["text"]


# --- names, places and values shared by the bilingual lines and the forms ---
MALE_NAMES = ["محمد", "أحمد", "خالد", "سعيد", "عبدالله", "راشد", "حمدان", "منصور", "سلطان", "علي", "حمد", "يوسف", "عمر", "فيصل"]
FEMALE_NAMES = ["فاطمة", "مريم", "عائشة", "شيخة", "نورة", "سارة", "ليلى", "موزة", "حصة", "آمنة", "هند", "لطيفة"]
FAMILY_NAMES = ["الكعبي", "المنصوري", "الظاهري", "النعيمي", "الشامسي", "المزروعي", "الكتبي", "السويدي", "المهيري", "الهاشمي", "البلوشي", "العامري"]
LATIN_NAMES = {"محمد": "Mohammed", "أحمد": "Ahmed", "خالد": "Khalid", "سعيد": "Saeed", "عبدالله": "Abdulla", "راشد": "Rashid",
               "حمدان": "Hamdan", "منصور": "Mansoor", "سلطان": "Sultan", "علي": "Ali", "حمد": "Hamad", "يوسف": "Yousef", "عمر": "Omar",
               "فيصل": "Faisal", "فاطمة": "Fatima", "مريم": "Mariam", "عائشة": "Aisha", "شيخة": "Shaikha", "نورة": "Noura", "سارة": "Sara",
               "ليلى": "Laila", "موزة": "Moza", "حصة": "Hessa", "آمنة": "Amna", "هند": "Hind", "لطيفة": "Latifa",
               "الكعبي": "Al Kaabi", "المنصوري": "Al Mansoori", "الظاهري": "Al Dhaheri", "النعيمي": "Al Nuaimi", "الشامسي": "Al Shamsi",
               "المزروعي": "Al Mazrouei", "الكتبي": "Al Ketbi", "السويدي": "Al Suwaidi", "المهيري": "Al Muhairi", "الهاشمي": "Al Hashemi",
               "البلوشي": "Al Balushi", "العامري": "Al Ameri"}
EMIRATES = ["أبوظبي", "دبي", "الشارقة", "عجمان", "أم القيوين", "رأس الخيمة", "الفجيرة"]
NATIONALITIES = ["الإمارات", "السعودية", "مصر", "الأردن", "الهند", "باكستان", "الفلبين", "سوريا", "لبنان", "المغرب", "بنغلاديش", "السودان"]
COMPANY_WORDS = {"النور": "Al Noor", "الخليج": "Al Khaleej", "الإمارات": "Emirates", "المستقبل": "Al Mustaqbal", "البركة": "Al Baraka",
                 "الصقر": "Al Saqr", "النخيل": "Al Nakheel", "الأفق": "Al Ufuq", "السلام": "Al Salam", "الواحة": "Al Waha"}
COMPANY_KINDS = [("للتجارة", "Trading LLC"), ("للمقاولات", "Contracting LLC"), ("للخدمات", "Services LLC"), ("للتقنية", "Technologies FZE")]
STREETS = ["شارع الكورنيش", "شارع الشيخ زايد", "شارع المطار", "شارع خليفة", "شارع حمدان", "شارع الوحدة"]
FORM_TITLES = [("استمارة طلب خدمة", "Service Request Form"), ("فاتورة", "Invoice"), ("طلب تسجيل", "Registration Application"),
               ("طلب تجديد رخصة", "Licence Renewal Request"), ("إيصال دفع", "Payment Receipt"), ("نموذج بيانات الموظف", "Employee Data Form")]
FIELD_LABELS = {   # JSON key -> (Arabic label, English label)
    "reference_number": ("رقم الطلب", "Reference No."), "date": ("التاريخ", "Date"), "full_name": ("الاسم الكامل", "Full Name"),
    "phone": ("رقم الهاتف", "Phone"), "emirate": ("الإمارة", "Emirate"), "amount_aed": ("المبلغ (درهم)", "Amount (AED)"),
    "emirates_id": ("رقم الهوية", "Emirates ID"), "nationality": ("الجنسية", "Nationality"), "email": ("البريد الإلكتروني", "Email"),
    "expiry_date": ("تاريخ الانتهاء", "Expiry Date"), "company": ("اسم الشركة", "Company"), "address": ("العنوان", "Address"),
}
CORE_FIELDS = ["reference_number", "date", "full_name", "phone", "emirate", "amount_aed"]   # v4's six


def form_instruction(keys):
    return ("Extract the fields from this form as JSON with keys: " + ", ".join(keys)
            + ". Use Western digits, dates as YYYY-MM-DD and phone numbers as digits only.")


def random_name(rng):
    first, middle, family = rng.choice(MALE_NAMES + FEMALE_NAMES), rng.choice(MALE_NAMES), rng.choice(FAMILY_NAMES)
    return f"{first} {middle} {family}", f"{LATIN_NAMES[first]} {LATIN_NAMES[middle]} {LATIN_NAMES[family]}"


def random_date(rng, year_from=2020, year_to=2027):
    year, month, day = rng.randint(year_from, year_to), rng.randint(1, 12), rng.randint(1, 28)
    answer = f"{year:04d}-{month:02d}-{day:02d}"
    printed = rng.choice([f"{day:02d}/{month:02d}/{year}", f"{day:02d}-{month:02d}-{year}", f"{day:02d}.{month:02d}.{year}", answer])
    return answer, printed


def field_value(rng, key, show):
    """(answer, printed) for one form field; `show` converts digits to Arabic-Indic when the form uses them."""
    if key == "reference_number":
        prefix, number = rng.choice(["SR", "INV", "REG", "REQ", "PAY"]), str(rng.randint(10000, 999999))
        return f"{prefix}-{number}", f"{prefix}-{show(number)}"
    if key in ("date", "expiry_date"):
        answer, printed = random_date(rng, *((2025, 2031) if key == "expiry_date" else (2020, 2027)))
        return answer, show(printed)
    if key == "full_name":
        arabic, latin = random_name(rng)
        return arabic, arabic
    if key == "phone":
        digits = "05" + "".join(str(rng.randint(0, 9)) for _ in range(8))
        printed = rng.choice([f"{digits[:3]} {digits[3:6]} {digits[6:]}", digits, f"+971 {digits[1:3]} {digits[3:6]} {digits[6:]}"])
        return digits, (show(printed) if rng.random() < 0.3 else printed)     # phone numbers are mostly in Western digits
    if key == "emirate":
        value = rng.choice(EMIRATES)
        return value, value
    if key == "amount_aed":
        amount = rng.randint(50, 50000) + rng.choice([0, 0.25, 0.5, 0.75])
        answer = f"{amount:.2f}"
        printed = rng.choice([f"{amount:,.2f}", f"AED {amount:,.2f}", f"{amount:,.2f} درهم"])
        return answer, show(printed)
    if key == "emirates_id":
        answer = f"784-{rng.randint(1950, 2010)}-{rng.randint(1000000, 9999999)}-{rng.randint(0, 9)}"
        return answer, answer                                                   # printed in Western digits, as on the card
    if key == "nationality":
        value = rng.choice(NATIONALITIES)
        return value, value
    if key == "email":
        arabic, latin = random_name(rng)
        parts = latin.lower().replace("al ", "al").split()
        value = rng.choice([f"{parts[0]}.{parts[-1]}", f"{parts[0][0]}{parts[-1]}", f"{parts[0]}{rng.randint(1, 99)}"]) + "@" + rng.choice(["example.ae", "mail.com", "company.ae", "gov.ae"])
        return value, value
    if key == "company":
        arabic_kind, english_kind = rng.choice(COMPANY_KINDS)
        value = f"شركة {rng.choice(list(COMPANY_WORDS))} {arabic_kind}"
        return value, value
    if key == "address":
        value = f"{rng.choice(EMIRATES)}، {rng.choice(STREETS)}، مبنى {rng.randint(1, 60)}"
        return value, show(value)
    raise KeyError(key)


def render_label_value(rng, label, value, style="page", min_size=22):
    """One line with an Arabic label on the right and a value drawn as its own left-to-right run beside it, the way a
    typesetter keeps a number in reading order on a form. (Drawn as one bidi string, the digit groups would reverse.)"""
    path = pick_font(rng, True, style)
    size = rng.randint(min_size, 40)
    font = get_font(path, size)
    probe = ImageDraw.Draw(Image.new("RGB", (10, 10)))
    value_font = font if (is_arabic(value) or re.search(r"[٠-٩]", value)) else get_font(rng.choice(MONO_FONTS + ENGLISH_BODY_FONTS), size)
    shown_value = typeset_ltr(value)
    label_w, value_w = text_width(probe, label, font), text_width(probe, shown_value, value_font)
    pad, gap = rng.randint(10, 32), int(size * 0.5)
    width = int(label_w + value_w + gap + 2 * pad + rng.randint(0, 60))
    bg, ink = colours(rng, style)
    img = background(rng, (width, 2 * pad + int(size * 1.7)), bg)
    draw = ImageDraw.Draw(img)
    draw_text(draw, pad, width - pad, pad, label, font, ink, "right")
    draw_text(draw, pad, width - pad - int(label_w) - gap, pad, shown_value, value_font, ink, "right")
    return img


def bilingual_line_example(rng, heldout=False):
    """Mixed Arabic and English on one line, the normal case on UAE documents."""
    kind = rng.random()
    show = (lambda s: s.translate(TO_ARABIC_DIGITS)) if rng.random() < 0.4 else (lambda s: s)
    if kind < 0.45:                                            # an Arabic label with a Latin or numeric value
        key = rng.choice(["reference_number", "phone", "emirates_id", "email", "amount_aed", "date", "full_name"])
        answer, printed = field_value(rng, key, show)
        if key == "full_name" and rng.random() < 0.7:
            printed = random_name(rng)[1]                      # the name in Latin letters
        label = FIELD_LABELS[key][0] + ":"
        text = f"{label} {printed}"
        if rng.random() < 0.5:                                 # typeset: the value as its own left-to-right run
            level = rng.choice([0, 1, 1])
            img = damage(render_label_value(rng, label, printed, rng.choice(["page", "page", "colour"])), rng, level)
            return img, OCR_INSTRUCTION, text
    elif kind < 0.65:                                          # Arabic and English labels side by side
        key = rng.choice(list(FIELD_LABELS))
        text = f"{FIELD_LABELS[key][0]} / {FIELD_LABELS[key][1]}"
    elif kind < 0.8:                                           # a place or company in both scripts
        if rng.random() < 0.5:
            arabic_kind, english_kind = rng.choice(COMPANY_KINDS)
            word = rng.choice(list(COMPANY_WORDS))
            text = f"شركة {word} {arabic_kind} - {COMPANY_WORDS[word]} {english_kind}"
        else:
            text = rng.choice(["أبوظبي - Abu Dhabi", "دبي - Dubai", "الشارقة - Sharjah", "عجمان - Ajman", "رأس الخيمة - Ras Al Khaimah",
                               "الفجيرة - Fujairah", "أم القيوين - Umm Al Quwain", "العين - Al Ain"])
    else:                                                      # an Arabic sentence with an English term inside
        words = make_line(rng, "msa", heldout, rng.randint(5, 12), extras=False).split()
        at = rng.randint(1, len(words) - 1)
        words.insert(at, rng.choice(LATIN_EXTRAS))
        text = " ".join(words)
    level = rng.choice([0, 1, 1])
    img = damage(render_lines([text], rng, rng.choice(["page", "page", "colour"]), min_size=22), rng, level)
    return img, OCR_INSTRUCTION, text


def text_width(draw, text, font):
    shown, options = prepare(text)
    return draw.textlength(shown, font=font, **options)


def fit_text(draw, x_left, x_right, y, text, font, color, align):
    """Draw text inside [x_left, x_right], shrinking the font when the text is wider than the space."""
    size = font.size
    while text_width(draw, text, font) > x_right - x_left and size > 14:
        size -= 2
        font = get_font(font.path, size)
    draw_text(draw, x_left, x_right, y, text, font, color, align)


def draw_stamp(draw, rng, box, text):
    x0, y0, x1, y1 = box
    colour = rng.choice([(150, 30, 30), (30, 60, 150), (40, 110, 60)])
    draw.ellipse(box, outline=colour, width=3)
    draw.ellipse((x0 + 8, y0 + 8, x1 - 8, y1 - 8), outline=colour, width=1)
    font = get_font(rng.choice(ARABIC_BODY_FONTS), max(14, (y1 - y0) // 6))
    draw_text(draw, x0, x1, (y0 + y1) // 2 - (y1 - y0) // 10, text, font, colour, "center")


def draw_signature(draw, rng, box):
    x0, y0, x1, y1 = box
    points = [(rng.uniform(x0, x1), rng.uniform(y0, y1)) for _ in range(rng.randint(6, 12))]
    points.sort()
    draw.line(points, fill=(20, 20, 90), width=2, joint="curve")


def draw_barcode(draw, rng, box):
    x0, y0, x1, y1 = box
    x = x0
    while x < x1:
        w = rng.choice([1, 2, 3])
        if rng.random() < 0.5:
            draw.rectangle((x, y0, x + w, y1), fill=(0, 0, 0))
        x += w + rng.choice([1, 2])


def form_example(rng, damage_level=1):
    """A made-up bilingual form. The answer is normalised: Western digits, ISO dates, digits-only phone numbers."""
    arabic_digits = rng.random() < 0.5
    show = (lambda s: s.translate(TO_ARABIC_DIGITS)) if arabic_digits else (lambda s: s)
    extra = [k for k in FIELD_LABELS if k not in CORE_FIELDS]
    keys = rng.sample(CORE_FIELDS, rng.randint(3, 6)) + rng.sample(extra, rng.randint(0, 3))
    rng.shuffle(keys)
    keys = keys[:rng.randint(4, 8)] if len(keys) > 4 else keys
    values = {k: field_value(rng, k, show) for k in keys}
    answer = {k: values[k][0] for k in keys}
    layout = rng.choice(["rows", "rows", "two_columns", "boxed", "table"])
    size = rng.randint(22, 32)
    arabic_font = get_font(rng.choice(ARABIC_BODY_FONTS), size)
    english_font = get_font(rng.choice(ENGLISH_BODY_FONTS), int(size * 0.8))
    value_fonts = {"ar": get_font(rng.choice(HAND_FONTS["ar"]), size) if rng.random() < 0.3 else arabic_font,
                   "en": get_font(rng.choice(HAND_FONTS["en"] if rng.random() < 0.3 else MONO_FONTS), size)}
    width, pad, row_height = rng.randint(700, 1200), rng.randint(24, 40), int(size * 2.1)
    rows = math.ceil(len(keys) / 2) if layout == "two_columns" else len(keys)
    header = 2 * row_height + (row_height if rng.random() < 0.5 else 0)            # title (+ a checkbox line)
    footer = int(row_height * 2.5) if rng.random() < 0.6 else row_height // 2        # stamp / signature / barcode
    img = Image.new("RGB", (width, 2 * pad + header + row_height * rows + footer), tuple(rng.randint(238, 255) for _ in range(3)))
    draw = ImageDraw.Draw(img)
    title_ar, title_en = rng.choice(FORM_TITLES)
    if rng.random() < 0.4:                                                           # a logo placeholder
        draw.rectangle((pad, pad, pad + size * 2, pad + size * 2), outline=(120, 120, 120), width=2)
    draw_text(draw, pad, width - pad, pad, title_ar, get_font(arabic_font.path, size + 6), (0, 0, 0), "right")
    draw_text(draw, pad + size * 3, width - pad, pad + size // 2, title_en, english_font, (0, 0, 0), "left")
    y = pad + 2 * row_height
    if header > 2 * row_height:                                                      # a decorative checkbox line (not in the JSON)
        options = rng.choice([("جديد", "تجديد", "إلغاء"), ("فرد", "شركة", "حكومة")])
        ticked, x = rng.randint(0, 2), width - pad
        for i, option in enumerate(options):
            box = (x - size, y + 4, x, y + 4 + size)
            draw.rectangle(box, outline=(60, 60, 60), width=2)
            if i == ticked:
                draw.line([(box[0] + 4, box[1] + size // 2), (box[0] + size // 2, box[3] - 4), (box[2] - 3, box[1] + 3)], fill=(0, 0, 0), width=3)
            draw_text(draw, pad, x - size - 10, y, option, arabic_font, (40, 40, 40), "right")
            x -= size * 6
        y += row_height
    ink = (0, 0, 0)
    typeset = rng.random() < 0.5                                                      # digit groups kept in reading order, or natural bidi order
    for i, key in enumerate(keys):
        label_ar, label_en = FIELD_LABELS[key]
        printed = values[key][1]
        value_font = value_fonts["ar" if (is_arabic(printed) or re.search(r"[٠-٩]", printed)) else "en"]
        if typeset:
            printed = typeset_ltr(printed)
        label_ar_w = text_width(draw, label_ar + ":", arabic_font)
        if layout == "two_columns":
            x_left, x_right = (pad, width // 2 - pad // 2) if i % 2 == 1 else (width // 2 + pad // 2, width - pad)
            row_y = y + (i // 2) * row_height
            draw_text(draw, x_left, x_right, row_y, label_ar + ":", arabic_font, (40, 40, 40), "right")
            fit_text(draw, x_left, x_right - int(label_ar_w) - 12, row_y, printed, value_font, ink, "right")
            continue
        row_y = y + i * row_height
        if layout == "table":
            divider = width - pad - int(label_ar_w) - 24
            draw.rectangle((pad, row_y - 6, width - pad, row_y + row_height - 6), outline=(140, 140, 140), width=1)
            draw.line([(divider, row_y - 6), (divider, row_y + row_height - 6)], fill=(140, 140, 140), width=1)
        elif layout == "rows":
            draw.line([(pad, row_y - 6), (width - pad, row_y - 6)], fill=(170, 170, 170), width=1)
        draw_text(draw, pad, width - pad, row_y, label_ar + ":", arabic_font, (40, 40, 40), "right")   # Arabic label
        area_left, area_right = pad, width - pad - int(label_ar_w) - 24                                 # where the value may go
        if layout != "table":
            draw_text(draw, pad, width - pad, row_y, label_en, english_font, (90, 90, 90), "left")         # English label
            area_left = pad + int(text_width(draw, label_en, english_font)) + 24
        if layout == "boxed":
            box = (area_left, row_y - 4, area_right, row_y + row_height - 10)
            draw.rectangle(box, outline=(90, 90, 90), width=2)
            fit_text(draw, box[0] + 8, box[2] - 8, row_y, printed, value_font, ink, "center")
        else:
            fit_text(draw, area_left, area_right, row_y, printed, value_font, ink, "center")              # the value
    y_footer = y + rows * row_height + 6
    if footer > row_height:
        if rng.random() < 0.6:
            draw_stamp(draw, rng, (pad, y_footer, pad + int(row_height * 2.2), y_footer + int(row_height * 2.2)), rng.choice(["مصدق", "مدفوع", "مستلم"]))
        if rng.random() < 0.6:
            draw_signature(draw, rng, (width // 2 - row_height * 2, y_footer, width // 2 + row_height * 2, y_footer + row_height * 2))
        if rng.random() < 0.5:
            draw_barcode(draw, rng, (width - pad - row_height * 4, y_footer + row_height // 2, width - pad, y_footer + row_height * 2))
    return damage(img, rng, damage_level), form_instruction(keys), json.dumps(answer, ensure_ascii=False)


class GeneratedData(torch.utils.data.Dataset):
    """n examples; example i is made by recipe(random.Random(seed, offset + i)), so the data is reproducible
    and a resumed run can start part-way through."""

    def __init__(self, recipe, n, seed, offset=0):
        self.recipe, self.n, self.seed, self.offset = recipe, n, seed, offset

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        return self.recipe(random.Random(self.seed * 1_000_003 + self.offset + i))


def show_examples(recipe, n=4):
    for i in range(n):
        img, instruction, answer = recipe(random.Random(99 + i))
        print(f"{img.size[0]}x{img.size[1]} px -> {answer[:110]!r}")
    return recipe(random.Random(99))[0]


# Fixed evaluation sets, identical for every stage
EVAL_SETS = {
    "arabic":       [text_line_example(random.Random(10_000 + i), "ar", 1, heldout=True) for i in range(N_EVAL)],
    "arabic_diac":  [text_line_example(random.Random(15_000 + i), "ar", 1, heldout=True, source="tashkeel") for i in range(N_EVAL)],
    "english":      [text_line_example(random.Random(20_000 + i), "en", 1, heldout=True) for i in range(N_EVAL)],
    "bilingual":    [bilingual_line_example(random.Random(25_000 + i), heldout=True) for i in range(N_EVAL)],
    "khatt":        [khatt_example(random.Random(40_000 + i), split="test") for i in range(N_EVAL)],
    "muharaf":      [muharaf_example(random.Random(45_000 + i), split="test") for i in range(N_EVAL)],
    "handwriting":  [iam_example(random.Random(50_000 + i), split="test") for i in range(N_EVAL)],
}
EVAL_FORMS = [form_example(random.Random(30_000 + i)) for i in range(N_EVAL)]
SET_NAMES = {"arabic": "Arabic (drawn, held-out text)", "arabic_diac": "Arabic with diacritics (drawn)",
             "english": "English (drawn, held-out text)", "bilingual": "Bilingual lines (drawn)",
             "khatt": "Arabic handwriting (KHATT test)", "muharaf": "Arabic handwriting (Muharaf test)",
             "handwriting": "English handwriting (IAM test)"}
print("evaluation sets:", ", ".join(f"{SET_NAMES[k]} {len(v)}" for k, v in EVAL_SETS.items()), f", forms {len(EVAL_FORMS)}")
EVAL_FORMS[0][0]

In [ ]:
# What the drawing styles look like: the four line styles, a diacritised line, a bilingual line and a form
samples = [render_lines([make_line(random.Random(7 + i), "msa", False, 9, True)], random.Random(7 + i), style)
           for i, style in enumerate(["page", "colour", "slide", "sign"])]
samples.append(render_lines([make_line(random.Random(3), "tashkeel", False, 8, False)], random.Random(3), "page"))
samples.append(bilingual_line_example(random.Random(5))[0])
sheet = Image.new("RGB", (max(im.width for im in samples) + 20, sum(im.height + 10 for im in samples) + 10), "white")
y = 10
for im in samples:
    sheet.paste(im, (10, y))
    y += im.height + 10
sheet

## E. Loading, batching and evaluation tools

- `load_model(stage)` loads the checkpoint saved by a stage (0 = the assembled model from part 1). The image size limits
  are passed as `size={"shortest_edge", "longest_edge"}`, the form transformers 5 keeps (`min_pixels`/`max_pixels` are
  deprecated).
- `make_batch(examples)` turns examples into model inputs. Each example becomes a chat (user = image + instruction,
  assistant = answer). **The loss is computed only on the answer**: every token up to and including the
  `<|im_start|>assistant` header, and all padding, gets the label -100, which the loss ignores.
- `read_images(...)` generates answers. **New in v5:** the token limit comes from the image, `max_tokens_for(kind, visual
  tokens)`: a line gets 32 tokens plus one per visual token (ceiling 512), a page the same with ceiling 640, a form 400.
  v4's fixed limits cut long diacritised lines (64 tokens in the speed benchmark) and let loops run; this limit grows
  with the image and never falls below what a line of that size can hold. Every output also goes through
  `OUTPUT_CHAR_MAP` (Farsi yeh and kaf become Arabic yeh and kaf), for every model alike.

**The measurements.** For each evaluation set:

| Measure | Meaning | Perfect |
| --- | --- | --- |
| **CER**, character error rate | character edits (insert, delete, replace) needed to turn the model's answer into the truth, divided by the number of characters in the truth | 0 |
| **WER**, word error rate | the same, counting whole words | 0 |
| **length** | characters in the model's answers divided by characters in the truth | 1.0 |
| **CER / WER without marks** (v5) | the same after removing diacritics and tatweel from both sides | 0 |
| **Arabic-normalised CER / WER** (v5) | the same after also unifying alef forms (أ إ آ → ا), ى → ي and ة → ه: the usual Arabic OCR normalisation, reported beside the raw score, never instead of it | 0 |

CER and WER can go above 1: if the truth is 30 characters and the model writes 120 characters of something else,
CER is about 3–4. The **length** column makes this visible. On the public sets a 95% bootstrap interval is given
for CER (resampling images), so differences between models can be read against the noise of a 200–500 image set.

Two losses are also measured on the evaluation images, without generating: **eval loss** with the real image, and
**no-image loss** with a blank white image of the same size. If they are equal, the model is guessing text from
the instruction and ignoring the picture. The gap between them is how much the model really reads the image.

In [ ]:
def processor_for(path, max_pixels=None):
    return AutoProcessor.from_pretrained(path, size={"shortest_edge": MIN_PIXELS, "longest_edge": max_pixels or MAX_PIXELS})


def load_model(stage_or_path, dtype=torch.float32, max_pixels=None):
    path = CHECKPOINTS[stage_or_path] if stage_or_path in CHECKPOINTS else stage_or_path
    if stage_or_path in CHECKPOINTS:
        assert os.path.isdir(path), f"{path} not found: run stage {stage_or_path} first"
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(path, dtype=dtype).to(DEVICE)
    processor = processor_for(path, max_pixels)
    print(f"loaded {path}: {count_parameters(model) / 1e9:.3f}B parameters")
    return model, processor


def count_parameters(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


def freeze_everything(model):
    for p in model.parameters():
        p.requires_grad = False


def report_trainable(model):
    n, total = count_parameters(model, trainable_only=True), count_parameters(model)
    print(f"trainable: {n / 1e6:,.1f}M of {total / 1e6:,.0f}M parameters ({100 * n / total:.2f}%)")


def chat(instruction, answer=None):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": instruction}]}]
    if answer is not None:
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return messages


def make_batch(examples, processor):
    tok = processor.tokenizer
    tok.padding_side = "right"
    texts = [processor.apply_chat_template(chat(instruction, answer), tokenize=False) for _, instruction, answer in examples]
    batch = processor(text=texts, images=[img for img, _, _ in examples], padding=True, return_tensors="pt")
    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100                       # ignore padding
    header = tok("<|im_start|>assistant\n", add_special_tokens=False).input_ids
    for row, ids in enumerate(batch["input_ids"].tolist()):
        starts = [j for j in range(len(ids) - len(header) + 1) if ids[j:j + len(header)] == header]
        labels[row, :starts[-1] + len(header)] = -100                 # ignore everything before the answer
    batch["labels"] = labels
    return batch


def amp():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP)


def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


TOKEN_LIMITS = {"line": (32, 1.0, 512), "page": (32, 1.0, 640), "form": (400, 0.0, 400)}   # kind -> (base, per visual token, ceiling)


def max_tokens_for(kind, visual_tokens):
    base, per_token, ceiling = TOKEN_LIMITS[kind]
    return 16 if QUICK_TEST else int(min(ceiling, base + per_token * visual_tokens))


def clean_output(text):
    return text.translate(OUTPUT_CHAR_MAP).strip()


@torch.no_grad()
def read_images(model, processor, examples, kind="line", batch_size=8, timings=None, decoding=None, max_new_tokens=None):
    """Generate answers. Left padding, so every prompt in a batch ends exactly where the answer starts.
    If `timings` is a list, (seconds, tokens generated) is appended for each batch."""
    tok = processor.tokenizer
    tok.padding_side = "left"
    was_training = model.training
    model.eval()
    model.config.use_cache = True
    settings = DECODING if decoding is None else decoding
    answers = []
    for i in range(0, len(examples), batch_size):
        part = examples[i:i + batch_size]
        texts = [processor.apply_chat_template(chat(instruction), tokenize=False, add_generation_prompt=True)
                 for _, instruction, _ in part]
        inputs = processor(text=texts, images=[img for img, _, _ in part], padding=True, return_tensors="pt").to(DEVICE)
        grid = inputs["image_grid_thw"]
        visual_tokens = int((grid[:, 0] * grid[:, 1] * grid[:, 2]).max()) // 4
        limit = max_new_tokens or max_tokens_for(kind, visual_tokens)
        sync()
        start = time.perf_counter()
        with amp():
            out = model.generate(**inputs, max_new_tokens=limit, do_sample=False, **settings)
        sync()
        new = out[:, inputs["input_ids"].shape[1]:]
        if timings is not None:
            timings.append((time.perf_counter() - start, int((new != tok.pad_token_id).sum())))
        answers += [clean_output(a) for a in processor.batch_decode(new, skip_special_tokens=True)]
    tok.padding_side = "right"
    model.config.use_cache = False
    model.train(was_training)
    return answers


ARABIC_LETTER_MAP = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ى": "ي", "ة": "ه"})


def normalise(text):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", text)).strip()


def normalise_arabic(text):
    return normalise(strip_marks(text)).translate(ARABIC_LETTER_MAP)


def edit_distance(a, b):
    """Works on strings (characters) and on lists (words)."""
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]


def error_rates(predictions, truths):
    """Per-image (edits, length) pairs for characters and words, and the totals."""
    chars = [(edit_distance(p, t), len(t)) for p, t in zip(predictions, truths)]
    words = [(edit_distance(p.split(), t.split()), len(t.split())) for p, t in zip(predictions, truths)]
    cer = sum(e for e, _ in chars) / max(1, sum(n for _, n in chars))
    wer = sum(e for e, _ in words) / max(1, sum(n for _, n in words))
    return cer, wer, chars


def bootstrap_interval(pairs, rounds=1000, seed=0):
    """95% interval for an error rate = sum(edits) / sum(lengths), resampling images."""
    if len(pairs) < 2:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    edits, lengths = np.array([e for e, _ in pairs], dtype="float64"), np.array([n for _, n in pairs], dtype="float64")
    samples = rng.integers(0, len(pairs), size=(rounds, len(pairs)))
    rates = edits[samples].sum(1) / np.maximum(1, lengths[samples].sum(1))
    return (float(np.percentile(rates, 2.5)), float(np.percentile(rates, 97.5)))


def text_scores(predictions, truths, interval=False):
    p, t = [normalise(x) for x in predictions], [normalise(x) for x in truths]
    cer, wer, pairs = error_rates(p, t)
    cer_plain, wer_plain, _ = error_rates([strip_marks(x) for x in p], [strip_marks(x) for x in t])
    cer_norm, wer_norm, _ = error_rates([normalise_arabic(x) for x in p], [normalise_arabic(x) for x in t])
    scores = {"cer": cer, "wer": wer, "length": sum(len(a) for a in p) / max(1, sum(len(b) for b in t)),
              "cer_no_marks": cer_plain, "wer_no_marks": wer_plain, "cer_norm": cer_norm, "wer_norm": wer_norm}
    if interval:
        scores["cer_ci"] = bootstrap_interval(pairs)
    return scores


def form_scores(predictions, examples):
    correct = valid = total = 0
    for prediction, (_, _, answer) in zip(predictions, examples):
        truth = json.loads(answer)
        try:
            got = json.loads(re.search(r"\{.*\}", prediction, re.S).group(0))
            valid += 1
        except Exception:
            got = {}
        total += len(truth)
        correct += sum(normalise(str(got.get(k, ""))).translate(TO_WESTERN_DIGITS) == normalise(truth[k]) for k in truth)
    return {"field_accuracy": correct / max(1, total), "valid_json": valid / max(1, len(examples))}


@torch.no_grad()
def eval_losses(model, processor, examples, batch_size=8):
    """Average answer loss with the real images, and with blank images of the same size."""
    was_training = model.training
    model.eval()
    result = {}
    for name, swap in [("eval_loss", False), ("no_image_loss", True)]:
        total = count = 0
        for i in range(0, len(examples), batch_size):
            part = examples[i:i + batch_size]
            if swap:
                part = [(Image.new("RGB", img.size, "white"), ins, ans) for img, ins, ans in part]
            batch = {k: v.to(DEVICE) for k, v in make_batch(part, processor).items()}
            with amp():
                loss = model(**batch).loss.float().item()
            total, count = total + loss * len(part), count + len(part)
        result[name] = total / count
    model.train(was_training)
    return result


def evaluate(model, processor, label, include_forms=False, n=None, show=True):
    """Score every evaluation set. n = how many images per set (None = all). Returns a flat dict of scores."""
    start = time.time()
    scores = eval_losses(model, processor, EVAL_SETS["arabic"][:n or N_CHECK] + EVAL_SETS["english"][:n or N_CHECK])
    for name, examples in EVAL_SETS.items():
        examples = examples[:n]
        predictions = read_images(model, processor, examples)
        for metric, value in text_scores(predictions, [a for _, _, a in examples]).items():
            scores[f"{name}_{metric}"] = value
        if show:
            print(f"   {SET_NAMES[name]}\n      truth: {examples[0][2]}\n      model: {predictions[0]}")
    if include_forms:
        examples = EVAL_FORMS[:n]
        predictions = read_images(model, processor, examples, kind="form", batch_size=4)
        scores.update(form_scores(predictions, examples))
        if show:
            print(f"   form\n      truth: {examples[0][2]}\n      model: {predictions[0]}")
    if show:
        print_scores(label, scores, (time.time() - start) / 60)
    return scores


def print_scores(label, scores, minutes=None):
    print(f"\n[{label}]  eval loss {scores['eval_loss']:.3f}  (no-image loss {scores['no_image_loss']:.3f})"
          + (f"   {minutes:.1f} min" if minutes is not None else ""))
    print(f"   {'set':34s} {'CER':>6s} {'WER':>6s} {'length':>7s} {'CER no marks':>13s} {'CER normalised':>15s}")
    for name in EVAL_SETS:
        print(f"   {SET_NAMES[name]:34s} {scores[name + '_cer']:6.3f} {scores[name + '_wer']:6.3f} {scores[name + '_length']:7.2f} "
              f"{scores[name + '_cer_no_marks']:13.3f} {scores[name + '_cer_norm']:15.3f}")
    if "field_accuracy" in scores:
        print(f"   forms: field accuracy {scores['field_accuracy']:.3f}, valid JSON {scores['valid_json']:.3f}")


def stage_start_scores(stage, model, processor, include_forms=False):
    """Scores before a stage trains. Saved to Drive, so a resumed stage does not measure them again."""
    path = os.path.join(BASE_DIR, f"{RUN_NAME}_stage{stage}_start_scores.json")
    if os.path.exists(path):
        scores = json.load(open(path))
        print_scores(f"before stage {stage} (saved earlier)", scores)
        return scores
    scores = evaluate(model, processor, f"before stage {stage}", include_forms)
    json.dump(scores, open(path, "w"), indent=1)
    return scores

## F. Training loop, adapters, progress checks and saving

`train(model, processor, recipe, stage, examples, micro_batch, accum, lr)`:

- feeds `micro_batch` examples at a time and adds up gradients over `accum` of them, so each optimizer step
  sees `micro_batch × accum` examples while GPU memory only holds `micro_batch`;
- uses mixed precision (bf16 on an L4) with gradient checkpointing to save memory;
- warms the learning rate up over the first 5% of steps, then lowers it along a cosine curve to a tenth; parameters
  trained in full (the last vision blocks) get `LR_FULL` instead of the stage's LoRA learning rate;
- skips a step whose loss is not a finite number and stops if that happens too often;
- **every `steps / CHECKS_PER_STAGE` steps** runs a quick check (eval loss, no-image loss, CER, WER, length on
  `N_CHECK` images per set) and saves its progress to Drive. If the session dies, running the same stage's cells
  again continues from the last check instead of starting over.

`attach_adapters(model, decoder)` puts LoRA adapters (rank 32, alpha 64, dropout 0.05) on the vision encoder's attention
and feed-forward layers and, when `decoder=True`, on the decoder's, keeps the projector training in full, and, **new in
v5**, unfreezes the last `VISION_FULL_BLOCKS` vision blocks completely (no adapters on those). Rank-32 adapters on every
block were v4's only way to change the vision encoder; full weights on the top blocks let it adapt more to Arabic
handwriting where the features are most task-specific, at a learning rate (`LR_FULL`) suited to full weights.

`save_and_check(model, processor, stage)` merges LoRA adapters into the weights (so every checkpoint is a plain
model), saves to Drive, reloads the saved copy and checks it gives the same output.

In [ ]:
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Which layers get LoRA adapters (matched against layer names, e.g. "model.visual.blocks.5.attn.qkv")
DECODER_LAYERS = r".*language_model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))$"


def vision_layers(model, full_blocks):
    """Regex for the vision blocks that get LoRA adapters: every block except the last `full_blocks`."""
    depth = len(base_of(model).model.visual.blocks)
    ids = "|".join(str(i) for i in range(depth - full_blocks)) if full_blocks < depth else r"\d+"
    return rf".*visual\.blocks\.({ids})\.(attn\.qkv|attn\.proj|mlp\.(gate_proj|up_proj|down_proj))$"


def base_of(model):
    return model.base_model.model if hasattr(model, "peft_config") else model


def attach_adapters(model, decoder, full_blocks=None):
    """Make a model trainable for stages 2-4: LoRA adapters (+ the projector in full, + the last vision blocks in full)."""
    full_blocks = VISION_FULL_BLOCKS if full_blocks is None else full_blocks
    freeze_everything(model)
    if FULL_FINETUNE:
        for p in model.model.visual.parameters():                   # vision encoder + projector (the projector sits inside visual)
            p.requires_grad = True
        if decoder:
            for p in model.model.language_model.parameters():
                p.requires_grad = True
    else:
        targets = vision_layers(model, full_blocks)
        if decoder:
            targets = f"({targets})|({DECODER_LAYERS})"
        lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
                          target_modules=targets, modules_to_save=["merger"])
        model = get_peft_model(model, lora)
        blocks = base_of(model).model.visual.blocks
        for block in list(blocks)[len(blocks) - full_blocks:]:       # the last blocks train in full
            for p in block.parameters():
                p.requires_grad = True
    report_trainable(model)
    return model


def param_groups(model, lr):
    """LoRA adapters and the projector at the stage's lr; weights trained in full (vision blocks, or everything on a big GPU) at LR_FULL."""
    adapter, full = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (adapter if ("lora_" in name or "merger" in name or "modules_to_save" in name) else full).append(p)
    groups = [{"params": adapter, "base_lr": lr}] if adapter else []
    if full:
        groups.append({"params": full, "base_lr": LR_FULL if not FULL_FINETUNE else lr})
    return groups


def progress_path(stage):
    return os.path.join(BASE_DIR, f"{RUN_NAME}_stage{stage}_in_progress.pt")


def train(model, processor, recipe, stage, examples, micro_batch, accum, lr, include_forms=False):
    if QUICK_TEST:
        micro_batch, accum = 2, 1
    per_step = micro_batch * accum
    steps = max(1, examples // per_step)
    check_every = max(1, steps // CHECKS_PER_STAGE)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()            # lets gradients reach adapters inside checkpointed layers
    model.config.use_cache = False
    model.train()
    params = {n: p for n, p in model.named_parameters() if p.requires_grad}
    optimizer = torch.optim.AdamW(param_groups(model, lr), lr=lr, weight_decay=0.0, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    warmup = max(1, steps // 20)

    def schedule(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, steps - warmup)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))   # cosine down to 10%

    # --- resume from saved progress, if this stage was interrupted ---
    start_step, history, skipped, minutes_before = 0, [], 0, 0.0
    if os.path.exists(progress_path(stage)):
        saved = torch.load(progress_path(stage), map_location="cpu", weights_only=False)
        result = model.load_state_dict(saved["weights"], strict=False)
        assert not result.unexpected_keys, f"saved progress does not match this model: {result.unexpected_keys[:3]}"
        optimizer.load_state_dict(saved["optimizer"])
        scaler.load_state_dict(saved["scaler"])
        start_step, history, skipped, minutes_before = saved["step"], saved["history"], saved["skipped"], saved["minutes"]
        print(f"resuming stage {stage} at step {start_step}/{steps} (saved progress from {minutes_before:.0f} min of training)")

    def save_progress(step, minutes):
        tmp = progress_path(stage) + ".tmp"
        torch.save({"weights": {n: p.detach().cpu() for n, p in params.items()}, "optimizer": optimizer.state_dict(),
                    "scaler": scaler.state_dict(), "step": step, "history": history, "skipped": skipped,
                    "minutes": minutes}, tmp)
        os.replace(tmp, progress_path(stage))

    def check(step, train_loss, minutes):
        scores = evaluate(model, processor, "", include_forms, n=N_CHECK, show=False)
        history.append({"step": step, "examples": step * per_step, "train_loss": train_loss, **scores})
        line = (f"  check @ step {step}: eval loss {scores['eval_loss']:.3f} (no image {scores['no_image_loss']:.3f}) | "
                f"Arabic CER {scores['arabic_cer']:.3f} WER {scores['arabic_wer']:.3f} len {scores['arabic_length']:.2f} | "
                f"English CER {scores['english_cer']:.3f} | bilingual CER {scores['bilingual_cer']:.3f} | "
                f"KHATT CER {scores['khatt_cer']:.3f} | Muharaf CER {scores['muharaf_cer']:.3f} | IAM CER {scores['handwriting_cer']:.3f} | "
                f"diacritised CER {scores['arabic_diac_cer']:.3f}")
        if "field_accuracy" in scores:
            line += f" | fields {scores['field_accuracy']:.3f}"
        print(line)
        save_progress(step, minutes)

    print(f"stage {stage}: {steps:,} steps x {per_step} examples = {steps * per_step:,} examples, "
          f"progress check every {check_every} steps")
    if start_step == 0:
        check(0, None, 0.0)

    batches = iter(DataLoader(GeneratedData(recipe, (steps - start_step) * per_step, seed=stage, offset=start_step * per_step),
                              batch_size=micro_batch, collate_fn=lambda ex: make_batch(ex, processor),
                              num_workers=0 if QUICK_TEST else 2))
    start, recent, since_check = time.time(), [], []
    for step in range(start_step, steps):
        optimizer.zero_grad(set_to_none=True)
        step_loss, finite = 0.0, True
        for _ in range(accum):
            batch = {k: v.to(DEVICE) for k, v in next(batches).items()}
            with amp():
                loss = model(**batch).loss / accum
            if not torch.isfinite(loss):
                finite = False
                break
            scaler.scale(loss).backward()
            step_loss += loss.item()
        if not finite:
            skipped += 1
            if skipped > max(5, steps // 10):
                raise RuntimeError("Too many non-finite losses: set USE_AMP = False in section A, or use an L4/A100.")
            continue
        for group in optimizer.param_groups:
            group["lr"] = group["base_lr"] * schedule(step)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(list(params.values()), 1.0)
        scaler.step(optimizer)
        scaler.update()
        recent.append(step_loss)
        since_check.append(step_loss)
        minutes = minutes_before + (time.time() - start) / 60
        if (step + 1) % (1 if QUICK_TEST else 50) == 0 or step + 1 == steps:
            done = step + 1 - start_step
            memory = f" | GPU peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if DEVICE.type == "cuda" else ""
            print(f"step {step + 1}/{steps} | loss {sum(recent) / len(recent):.4f} | lr {lr * schedule(step):.1e} | "
                  f"{minutes:.1f} min, about {(time.time() - start) / 60 / done * (steps - step - 1):.0f} min left{memory}")
            recent = []
        if (step + 1) % check_every == 0 or step + 1 == steps:
            check(step + 1, sum(since_check) / max(1, len(since_check)), minutes)
            since_check = []
    minutes = minutes_before + (time.time() - start) / 60
    print(f"training done in {minutes:.1f} min ({skipped} steps skipped)")
    return {"steps": steps, "examples": steps * per_step, "minutes": round(minutes, 1), "skipped_steps": skipped,
            "history": history}


def plot_history(history, title, save_as=None):
    steps = [h["step"] for h in history]
    fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4))
    trained = [(h["step"], h["train_loss"]) for h in history if h["train_loss"] is not None]
    if trained:
        left.plot(*zip(*trained), label="training loss", marker="o")
    left.plot(steps, [h["eval_loss"] for h in history], label="eval loss (real image)", marker="o")
    left.plot(steps, [h["no_image_loss"] for h in history], label="eval loss (blank image)", linestyle="--")
    left.set_xlabel("step"), left.set_ylabel("loss"), left.set_title("loss"), left.legend()
    for name, style in [("arabic", "-"), ("arabic_diac", "--"), ("english", "-"), ("bilingual", "-."), ("khatt", ":"), ("muharaf", ":"), ("handwriting", ":")]:
        right.plot(steps, [h[f"{name}_cer"] for h in history], style, marker="o", label=f"CER {SET_NAMES[name]}")
    right.plot(steps, [h["arabic_wer"] for h in history], "--", marker="x", label="WER Arabic (drawn)")
    if "field_accuracy" in history[-1]:
        right.plot(steps, [1 - h["field_accuracy"] for h in history], "-.", marker="s", label="form fields wrong")
    right.set_xlabel("step"), right.set_ylabel("error rate (0 = perfect)"), right.set_title("error rates")
    right.set_ylim(0, min(2.0, max(1.05, max(h["arabic_cer"] for h in history) * 1.05)))
    right.legend(fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    if save_as:
        fig.savefig(save_as, dpi=150)
        print(f"chart saved to {save_as}")
    plt.show()


def save_and_check(model, processor, stage):
    if hasattr(model, "merge_and_unload"):
        model = model.merge_and_unload()           # fold LoRA adapters into the weights
    model.config.use_cache = True
    path = CHECKPOINTS[stage]
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    processor.save_pretrained(path)
    check = {k: v.to(DEVICE) for k, v in make_batch([EVAL_SETS["arabic"][0]], processor).items()}
    model.eval()
    with torch.no_grad():
        before_save = model(**check).logits.float()
    reloaded = Qwen2_5_VLForConditionalGeneration.from_pretrained(path, dtype=torch.float32).to(DEVICE).eval()
    with torch.no_grad():
        after_load = reloaded(**check).logits.float()
    diff = (before_save - after_load).abs().max().item()
    assert diff < 1e-3 * max(1.0, before_save.abs().max().item()), f"saved checkpoint differs (max {diff})"
    print(f"saved to {path}; reload check passed (max logit difference {diff:.1e})")
    del reloaded
    if os.path.exists(progress_path(stage)):
        os.remove(progress_path(stage))            # the stage is finished; its resume file is no longer needed
    if not KEEP_ALL_CHECKPOINTS and stage in (2, 3, 4):
        shutil.rmtree(CHECKPOINTS[stage - 1], ignore_errors=True)
        print(f"deleted {CHECKPOINTS[stage - 1]} to save Drive space")
    return model


def results_file():
    path = os.path.join(BASE_DIR, f"results_{RUN_NAME}.json")
    return path, (json.load(open(path)) if os.path.exists(path) else {})


def save_results(**updates):
    path, results = results_file()
    results.update(updates)
    json.dump(results, open(path, "w"), indent=1, ensure_ascii=False)
    return path


def record(stage, name, before, after, run):
    path, results = results_file()
    results[str(stage)] = {"name": name, "before": before, "after": after, **run}
    json.dump(results, open(path, "w"), indent=1, ensure_ascii=False)
    plot_history(run["history"], f"Stage {stage}: {name}", os.path.join(BASE_DIR, f"{RUN_NAME}_stage{stage}_curves.png"))
    columns = [("arabic_cer", "Ar CER"), ("arabic_diac_cer", "diac CER"), ("bilingual_cer", "biling CER"), ("khatt_cer", "KHATT CER"),
               ("muharaf_cer", "Muharaf CER"), ("english_cer", "En CER"), ("handwriting_cer", "IAM CER"), ("field_accuracy", "fields")]
    print(f"\n{'stage':26s}" + "".join(f"{label:>14s}" for _, label in columns))
    for s, r in sorted((k, v) for k, v in results.items() if k.isdigit()):
        cell = lambda k: f"{r['before'][k]:.2f} -> {r['after'][k]:.2f}" if k in r["after"] and k in r["before"] else ""
        print(f"{s + '. ' + r['name']:26s}" + "".join(f"{cell(k):>14s}" for k, _ in columns))


def free_memory():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

---
# Stage 0b (optional): pretrain the decoder on Arabic text

**Runs only when `PRETRAIN_DECODER = True`.** Stages 1 and 2 then train again, because the decoder the projector
was aligned to has changed; the switch sets `REUSE_STAGES_1_AND_2 = False` for you.

**Why.** The v4 model's remaining errors on printed classical text (PATS) and slides (ISI-PPT) are plausible wrong
words, and one PATS output even used the Farsi form of yeh. Qwen2.5-0.5B knows some Arabic, but little by the
standards of a text model, and OCR leans on the language model whenever a glyph is ambiguous. Continued pretraining
on in-domain text is the standard way to strengthen that prior before multimodal training. It is text only, so it is
cheap: about 2 hours on an L4 for 30 million tokens.

**What trains.** The whole decoder (`model.language_model`, 494M parameters; the output layer is tied to the
embeddings). The vision encoder and projector are untouched.

**Data.** The same sentence pools as the drawn lines: 70% modern standard Arabic (Wikipedia), 10% diacritised
sentences (so the model keeps a prior for tashkeel), 20% English (Wikipedia) so English is not forgotten. Sentences are
shuffled and packed into 1,024-token sequences. Held-out Arabic and English perplexity is reported before and after.

**Settings.** AdamW, learning rate 3e-5 with 5% warm-up and cosine decay to a tenth, 16 sequences of 1,024 tokens
per step, bf16, gradient checkpointing. Progress is saved at each check, like the other stages.

In [ ]:
def pretrain_decoder(model, processor, tokens, lr=3e-5, seq_len=1024, micro_batch=4, accum=4):
    if QUICK_TEST:
        micro_batch, accum = 1, 1
    tok = processor.tokenizer
    eos = tok.eos_token_id if tok.eos_token_id is not None else tok.pad_token_id
    rng = random.Random(11)

    def pack(sentences_by_share, n_tokens, seed):
        """Shuffle sentences from each pool in the given shares and pack them into seq_len-token rows. If the pools run
        out before n_tokens (only in small test runs), they are cycled."""
        r = random.Random(seed)
        picked = []
        for pool, share in sentences_by_share:
            k = max(1, int(len(pool) * min(1.0, share)))
            picked += r.sample(pool, min(k, len(pool)))
        r.shuffle(picked)
        wanted = max(1, n_tokens // seq_len)
        ids, rows, passes = [], [], 0
        while len(rows) < wanted and passes < 50:
            for i in range(0, len(picked), 256):
                for sentence_ids in tok(picked[i:i + 256], add_special_tokens=False).input_ids:
                    ids += sentence_ids + [eos]
                while len(ids) >= seq_len:
                    rows.append(ids[:seq_len])
                    ids = ids[seq_len:]
                if len(rows) >= wanted:
                    break
            passes += 1
        return torch.tensor(rows[:wanted], dtype=torch.long)

    # shares are fractions of each pool large enough to give the token budget; the pools are far bigger than needed
    words_needed = tokens / 3.0                                           # about 3 tokens per word
    msa, tashkeel, en = POOLS["msa"][0], POOLS["tashkeel"][0], POOLS["en"][0]
    share = lambda pool, fraction: min(1.0, fraction * words_needed / max(1, sum(len(s.split()) for s in pool[:2000]) / 2000 * len(pool)))
    train_rows = pack([(msa, share(msa, 0.7)), (tashkeel, share(tashkeel, 0.1)), (en, share(en, 0.2))], tokens, seed=1)
    held = {"arabic": pack([(POOLS["msa"][1], 0.05), (POOLS["tashkeel"][1], 0.05)], 64 * seq_len if not QUICK_TEST else 2 * seq_len, seed=2),
            "english": pack([(POOLS["en"][1], 0.05)], 32 * seq_len if not QUICK_TEST else 2 * seq_len, seed=3)}
    print(f"stage 0b: {len(train_rows):,} rows x {seq_len} tokens = {len(train_rows) * seq_len / 1e6:.1f}M tokens; "
          f"held-out Arabic {len(held['arabic'])} rows, English {len(held['english'])} rows")

    @torch.no_grad()
    def perplexity(rows):
        if len(rows) == 0:
            return float("nan")
        model.eval()
        total, count = 0.0, 0
        for i in range(0, len(rows), micro_batch):
            batch = rows[i:i + micro_batch].to(DEVICE)
            with amp():
                loss = model(input_ids=batch, labels=batch).loss.float().item()
            total, count = total + loss * len(batch), count + len(batch)
        model.train()
        return math.exp(total / max(1, count))

    freeze_everything(model)
    for p in model.model.language_model.parameters():
        p.requires_grad = True
    model.lm_head.weight.requires_grad = True
    report_trainable(model)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    model.train()
    params = {n: p for n, p in model.named_parameters() if p.requires_grad}
    optimizer = torch.optim.AdamW(params.values(), lr=lr, weight_decay=0.01, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE == torch.float16)
    steps = len(train_rows) // (micro_batch * accum)
    assert steps >= 1, f"too few pretraining rows ({len(train_rows)}) for {micro_batch} x {accum} per step: raise DECODER_PRETRAIN_TOKENS"
    warmup, check_every = max(1, steps // 20), max(1, steps // CHECKS_PER_STAGE)
    stage = "0b"
    start_step, history, minutes_before = 0, [], 0.0
    if os.path.exists(progress_path(stage)):
        saved = torch.load(progress_path(stage), map_location="cpu", weights_only=False)
        model.load_state_dict(saved["weights"], strict=False)
        optimizer.load_state_dict(saved["optimizer"])
        start_step, history, minutes_before = saved["step"], saved["history"], saved["minutes"]
        print(f"resuming stage 0b at step {start_step}/{steps}")
    before = {k: perplexity(v) for k, v in held.items()} if start_step == 0 else history[0]["perplexity"]
    if start_step == 0:
        history.append({"step": 0, "tokens": 0, "train_loss": None, "perplexity": before})
        print(f"  held-out perplexity before: Arabic {before['arabic']:.2f}, English {before['english']:.2f}")
    order = torch.randperm(len(train_rows), generator=torch.Generator().manual_seed(0))
    start, recent = time.time(), []
    for step in range(start_step, steps):
        factor = (step + 1) / warmup if step < warmup else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, steps - warmup)))
        for group in optimizer.param_groups:
            group["lr"] = lr * factor
        optimizer.zero_grad(set_to_none=True)
        step_loss = 0.0
        for a in range(accum):
            idx = order[(step * accum + a) * micro_batch:(step * accum + a + 1) * micro_batch]
            batch = train_rows[idx].to(DEVICE)
            with amp():
                loss = model(input_ids=batch, labels=batch).loss / accum
            scaler.scale(loss).backward()
            step_loss += loss.item()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(list(params.values()), 1.0)
        scaler.step(optimizer)
        scaler.update()
        recent.append(step_loss)
        minutes = minutes_before + (time.time() - start) / 60
        if (step + 1) % (1 if QUICK_TEST else 50) == 0 or step + 1 == steps:
            memory = f" | GPU peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if DEVICE.type == "cuda" else ""
            print(f"step {step + 1}/{steps} | loss {sum(recent) / len(recent):.4f} | lr {lr * factor:.1e} | {minutes:.1f} min{memory}")
            recent = []
        if (step + 1) % check_every == 0 or step + 1 == steps:
            ppl = {k: perplexity(v) for k, v in held.items()}
            history.append({"step": step + 1, "tokens": (step + 1) * micro_batch * accum * seq_len, "train_loss": step_loss, "perplexity": ppl})
            print(f"  check @ step {step + 1}: held-out perplexity Arabic {ppl['arabic']:.2f}, English {ppl['english']:.2f}")
            tmp = progress_path(stage) + ".tmp"
            torch.save({"weights": {n: p.detach().cpu() for n, p in params.items()}, "optimizer": optimizer.state_dict(),
                        "step": step + 1, "history": history, "minutes": minutes}, tmp)
            os.replace(tmp, progress_path(stage))
    after = history[-1]["perplexity"]
    print(f"stage 0b done in {minutes_before + (time.time() - start) / 60:.1f} min: perplexity Arabic {before['arabic']:.2f} -> {after['arabic']:.2f}, "
          f"English {before['english']:.2f} -> {after['english']:.2f}")
    return {"tokens": len(train_rows) * seq_len, "steps": steps, "history": history, "perplexity_before": before, "perplexity_after": after}


if PRETRAIN_DECODER:
    if os.path.isdir(CHECKPOINTS["0b"]) and not os.path.exists(progress_path("0b")):
        print(f"stage 0b already done: {CHECKPOINTS['0b']}")
    else:
        model, processor = load_model(0)
        stage0b_run = pretrain_decoder(model, processor, DECODER_PRETRAIN_TOKENS)
        model = save_and_check(model, processor, stage="0b")
        save_results(**{"0b": {"name": "decoder: Arabic text pretraining", **stage0b_run}})
        del model
        free_memory()
else:
    print("stage 0b skipped (PRETRAIN_DECODER = False): the decoder is used as shipped")

---
# Stage 1: projector alignment

**Skipped when `REUSE_STAGES_1_AND_2 = True`** (the default without decoder pretraining); the v3 stage 1 checkpoint is used.

**Goal.** The decoder has never seen visual tokens. This stage teaches the projector to map image features into
the decoder's embedding space, so the decoder can "read" them like words.

**What trains.** Only the projector (`model.visual.merger`, about 31M parameters). The vision encoder and the
decoder stay exactly as they came (from Qwen, or from stage 0b).

**Data (30,000 examples).** Clean drawn lines from Wikipedia sentences, 70% Arabic and 30% English.

**The three fixes from v3** stay: the projector's norm and first layer are copied from Qwen2.5-VL-3B's trained
projector (only the last layer, 5,120 → 896, starts random); the random last layer is rescaled so visual tokens
have the same typical size as the decoder's word embeddings; and 30,000 examples instead of v2's 3,200.

**Expect.** The gap between *eval loss* and *no-image loss* should open up within the first checks. If it stays near
zero after a third of the stage, the image is not getting through: stop and share the printed checks.

In [ ]:
if REUSE_STAGES_1_AND_2:
    print(f'skipped: {RUN_NAME} reuses the v3 checkpoint in {CHECKPOINTS[1]} and {CHECKPOINTS[2]}')
else:
    def stage1_example(rng):
        if rng.random() < 0.7:
            return text_line_example(rng, "ar", damage_level=0)
        return text_line_example(rng, "en", damage_level=0)


    show_examples(stage1_example)

In [ ]:
if REUSE_STAGES_1_AND_2:
    print(f'skipped: {RUN_NAME} reuses the v3 checkpoint in {CHECKPOINTS[1]} and {CHECKPOINTS[2]}')
else:
    model, processor = load_model("0b" if PRETRAIN_DECODER else 0)      # the assembled model, or the pretrained decoder
    merger = model.model.visual.merger


    def read_remote_tensors(repo, wanted):
        """Read a few named tensors from a safetensors checkpoint on the Hugging Face Hub (or a local folder) without
        downloading whole files: read the file header, then only the bytes of the wanted tensors."""
        from huggingface_hub import HfFileSystem, hf_hub_download
        local = os.path.isdir(repo)
        index = os.path.join(repo, "model.safetensors.index.json") if local else hf_hub_download(repo, "model.safetensors.index.json")
        weight_map = json.load(open(index))["weight_map"]
        opener = open if local else HfFileSystem().open
        tensors = {}
        for shard in sorted({weight_map[name] for name in wanted}):
            with opener(os.path.join(repo, shard) if local else f"{repo}/{shard}", "rb") as f:
                header_size = int.from_bytes(f.read(8), "little")
                header = json.loads(f.read(header_size))
                for name in wanted:
                    if weight_map[name] != shard:
                        continue
                    info = header[name]
                    begin, end = info["data_offsets"]
                    f.seek(8 + header_size + begin)
                    dtype = {"BF16": torch.bfloat16, "F16": torch.float16, "F32": torch.float32}[info["dtype"]]
                    tensors[name] = torch.frombuffer(bytearray(f.read(end - begin)), dtype=dtype).reshape(info["shape"]).clone()
        return tensors


    resuming = os.path.exists(progress_path(1))
    if WARM_START_PROJECTOR and not resuming:
        try:
            names = {"ln_q.weight": "visual.merger.ln_q.weight", "mlp.0.weight": "visual.merger.mlp.0.weight",
                     "mlp.0.bias": "visual.merger.mlp.0.bias"}
            qwen = read_remote_tensors(BASE_VL_ID, list(names.values()))
            own = dict(merger.named_parameters())
            with torch.no_grad():
                for ours, theirs in names.items():
                    assert own[ours].shape == qwen[theirs].shape, (ours, own[ours].shape, qwen[theirs].shape)
                    own[ours].copy_(qwen[theirs].float())
            print(f"projector: norm and first layer copied from {BASE_VL_ID}; the last layer stays random")
        except Exception as e:
            print(f"warm start skipped ({type(e).__name__}: {e}); the projector starts fully random")


    @torch.no_grad()
    def match_visual_token_scale(model, processor, examples):
        """Rescale the projector's last layer so visual tokens have the same average size (RMS) as the decoder's word
        embeddings for the answer text."""
        captured = []
        hook = model.model.visual.merger.register_forward_hook(lambda module, inputs, output: captured.append(output.float()))
        batch = {k: v.to(DEVICE) for k, v in make_batch(examples, processor).items()}
        model(**batch)
        hook.remove()
        visual_rms = captured[0].pow(2).mean().sqrt().item()
        answer_ids = batch["input_ids"][batch["labels"] != -100]
        word_rms = model.get_input_embeddings().weight[answer_ids].float().pow(2).mean().sqrt().item()
        last = model.model.visual.merger.mlp[2]
        last.weight.mul_(word_rms / visual_rms)
        last.bias.mul_(word_rms / visual_rms)
        print(f"visual token size {visual_rms:.4f} -> {word_rms:.4f} (word embedding size); last layer scaled by "
              f"{word_rms / visual_rms:.4f}")


    if not resuming:
        model.eval()
        match_visual_token_scale(model, processor, [stage1_example(random.Random(7_000 + i)) for i in range(8)])

    freeze_everything(model)
    for p in merger.parameters():                          # unfreeze only the projector
        p.requires_grad = True
    report_trainable(model)

    stage1_before = stage_start_scores(1, model, processor)

In [ ]:
if REUSE_STAGES_1_AND_2:
    print(f'skipped: {RUN_NAME} reuses the v3 checkpoint in {CHECKPOINTS[1]} and {CHECKPOINTS[2]}')
else:
    stage1_run = train(model, processor, stage1_example, stage=1, examples=STAGE_EXAMPLES[1],
                       micro_batch=8, accum=1, lr=1e-3)

In [ ]:
if REUSE_STAGES_1_AND_2:
    print(f'skipped: {RUN_NAME} reuses the v3 checkpoint in {CHECKPOINTS[1]} and {CHECKPOINTS[2]}')
else:
    stage1_after = evaluate(model, processor, "after stage 1")
    model = save_and_check(model, processor, stage=1)
    record(1, "align: projector", stage1_before, stage1_after, stage1_run)
    del model
    free_memory()

---
# Stage 2: vision encoder on general text

**Skipped when `REUSE_STAGES_1_AND_2 = True`**; the v3 stage 2 checkpoint is used.

**Goal.** Make the vision encoder better at reading text of any kind, including handwriting and damaged images,
before specialising in Arabic.

**What trains.** The vision encoder and the projector. The decoder stays frozen, so its language ability is kept.
On a GPU under 40 GB the vision encoder is trained with LoRA (plus the last `VISION_FULL_BLOCKS` blocks in full),
while the projector keeps training in full.

**Data (25,000 examples).**

| Share | Source |
| --- | --- |
| 25% | real handwritten English lines (IAM) |
| 15% | real handwritten Arabic lines (KHATT) |
| 15% | real handwritten Arabic lines (Muharaf) |
| 22% | drawn English lines, light damage |
| 23% | drawn Arabic lines, light damage |

**Expect.** English and handwriting error rates should fall the most here.

In [ ]:
if REUSE_STAGES_1_AND_2:
    print(f'skipped: {RUN_NAME} reuses the v3 checkpoint in {CHECKPOINTS[1]} and {CHECKPOINTS[2]}')
else:
    def stage2_example(rng):
        x = rng.random()
        if x < 0.25:
            return iam_example(rng)
        if x < 0.40:
            return khatt_example(rng)
        if x < 0.55:
            return muharaf_example(rng)
        if x < 0.77:
            return text_line_example(rng, "en", damage_level=1)
        return text_line_example(rng, "ar", damage_level=1)


    show_examples(stage2_example)

In [ ]:
if REUSE_STAGES_1_AND_2:
    print(f'skipped: {RUN_NAME} reuses the v3 checkpoint in {CHECKPOINTS[1]} and {CHECKPOINTS[2]}')
else:
    model, processor = load_model(1)
    model = attach_adapters(model, decoder=False)
    stage2_before = stage_start_scores(2, model, processor)

In [ ]:
if REUSE_STAGES_1_AND_2:
    print(f'skipped: {RUN_NAME} reuses the v3 checkpoint in {CHECKPOINTS[1]} and {CHECKPOINTS[2]}')
else:
    stage2_run = train(model, processor, stage2_example, stage=2, examples=STAGE_EXAMPLES[2],
                       micro_batch=4, accum=2, lr=2e-5 if FULL_FINETUNE else 2e-4)

In [ ]:
if REUSE_STAGES_1_AND_2:
    print(f'skipped: {RUN_NAME} reuses the v3 checkpoint in {CHECKPOINTS[1]} and {CHECKPOINTS[2]}')
else:
    stage2_after = evaluate(model, processor, "after stage 2")
    model = save_and_check(model, processor, stage=2)
    record(2, "vision: general text", stage2_before, stage2_after, stage2_run)
    del model
    free_memory()

---
# Stage 3: Arabic

**Goal.** Make the model good at Arabic script in realistic conditions. In Grab's experiments, skipping their
equivalent of this stage cost up to 10% on non-Latin documents.

**What trains.** The vision encoder (LoRA on the first blocks, the last `VISION_FULL_BLOCKS` in full) and the
projector, **plus LoRA adapters on the decoder** when `DECODER_LORA_IN_STAGE3 = True` (the default). Grab kept its
decoder frozen until the last stage, but its decoder was already strong in the target language; ours is a 0.5B model
that has never written Arabic transcriptions from images, so letting it adapt here is the faster route.

**Data (90,000 examples).** Every drawn line has 1–20 words, one of up to 39 Arabic fonts and one of the four
styles; damage is mixed from none to heavy. Against v4: the handwriting share is spread over KHATT (now thousands of
lines) and Muharaf, bilingual lines and SynthDoG pages are new, English handwriting is up from 8% to 10%.

| Share | Source | Aimed at |
| --- | --- | --- |
| 24% | Arabic lines from Wikipedia (modern standard Arabic), digits and Latin words mixed in | everyday printed text |
| 12% | diacritised Arabic lines | text with tashkeel (SythenAR) |
| 7% | the same kind of text with diacritics removed, plain instruction | classical, book-style text (PATS) |
| 6% | blocks of 2–5 lines | paragraphs and slides (ISI-PPT) |
| 9% | real Arabic handwriting (KHATT) | KHATT, OnlineKhatt |
| 9% | real Arabic handwriting (Muharaf, historical) | KHATT, OnlineKhatt, HistoryAr, Muharaf |
| 4% | 1–3 words in a sign style, tilted | street photos (EvArEST) |
| 5% | bilingual lines: Arabic labels with English or numeric values | UAE documents |
| 10% | English lines | keeping English |
| 10% | real English handwriting (IAM) | IAM |
| 4% | SynthDoG English pages (when prepared; otherwise English lines) | printed English on paper |

**Expect.** Drawn-set scores go up a little at first (the images are harder now), then down. The numbers that
matter are KHATT, Muharaf and the diacritised set in the checks, and the public benchmark at the end.

In [ ]:
def stage3_example(rng):
    x, level = rng.random(), rng.choice([0, 1, 1, 2])
    if x < 0.24:
        return text_line_example(rng, "ar", level, source="msa")
    if x < 0.36:
        return text_line_example(rng, "ar", level, source="tashkeel")
    if x < 0.43:
        return text_line_example(rng, "ar", level, source="classical")
    if x < 0.49:
        return text_line_example(rng, "ar", rng.choice([0, 1]), max_lines=5, source=rng.choice(["msa", "classical"]))
    if x < 0.58:
        return khatt_example(rng)
    if x < 0.67:
        return muharaf_example(rng)
    if x < 0.71:
        return scene_word_example(rng)
    if x < 0.76:
        return bilingual_line_example(rng)
    if x < 0.86:
        return text_line_example(rng, "en", level)
    if x < 0.96:
        return iam_example(rng)
    return synthdog_example(rng) if SYNTHDOG is not None else text_line_example(rng, "en", level, max_lines=4)


show_examples(stage3_example)

In [ ]:
model, processor = load_model(2)
model = attach_adapters(model, decoder=DECODER_LORA_IN_STAGE3)
stage3_before = stage_start_scores(3, model, processor)

In [ ]:
stage3_run = train(model, processor, stage3_example, stage=3, examples=STAGE_EXAMPLES[3],
                   micro_batch=4, accum=2, lr=1e-5 if FULL_FINETUNE else 2e-4)

In [ ]:
stage3_after = evaluate(model, processor, "after stage 3")
model = save_and_check(model, processor, stage=3)
record(3, "Arabic", stage3_before, stage3_after, stage3_run)
del model
free_memory()

---
# Stage 4: task fine-tuning (forms → JSON)

**Goal.** Teach the whole model the actual job: read a bilingual form and return its fields as JSON, with values
normalised (Western digits, dates as YYYY-MM-DD, phone as digits only). The v5 forms vary in fields, order and layout,
so a model that scores well here has learned to read labels, not a template.

**What trains.** Everything: vision encoder (LoRA + the last blocks in full), projector and decoder (LoRA).

**Data (16,000 examples).** 40% forms; the other 60% is the stage 3 mix, so the broader reading skills from
stage 3 are kept. The learning rate is `STAGE4_LR` (1e-4): in v4, stage 4 at 2e-4 cost a little handwriting accuracy
(KHATT CER 0.22 → 0.25 on 48 images), and the stage-3 checkpoint is scored on the public sets too, so you can pick
the better OCR model for a product that needs no JSON.

**Expect.** Most forms should come back as valid JSON. Field accuracy will not reach v4's 1.000 at once: the forms are
harder and some fields (addresses, company names) are longer.

In [ ]:
def stage4_example(rng):
    if rng.random() < 0.40:
        return form_example(rng, damage_level=rng.choice([0, 1, 1, 2]))
    return stage3_example(rng)


show_examples(stage4_example)

In [ ]:
model, processor = load_model(3)
model = attach_adapters(model, decoder=True)
stage4_before = stage_start_scores(4, model, processor, include_forms=True)

In [ ]:
stage4_run = train(model, processor, stage4_example, stage=4, examples=STAGE_EXAMPLES[4],
                   micro_batch=2, accum=4, lr=1e-5 if FULL_FINETUNE else STAGE4_LR, include_forms=True)

In [ ]:
stage4_after = evaluate(model, processor, "after stage 4", include_forms=True)
model = save_and_check(model, processor, stage=4)
record(4, "task: forms to JSON", stage4_before, stage4_after, stage4_run)
del model
free_memory()

---
# Results 1: the whole training run on one timeline
All stages' progress checks, one after another, measured in training examples seen. Run it any time; it reads
`results_v5.json` from Drive. (v4's version of this chart had an empty legend because it only labelled stage 1,
which v4 did not run; this one labels the first stage it finds.)

In [ ]:
_, results = results_file()
fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4.5))
offset = 0
stages = sorted(k for k in results if k.isdigit() and "history" in results[k])
for s in stages:
    history = results[s]["history"]
    x = [offset + h["examples"] for h in history]
    label = (lambda text: text if s == stages[0] else None)
    left.plot(x, [h["arabic_cer"] for h in history], "o-", color="tab:green", label=label("Arabic CER"))
    left.plot(x, [h["arabic_diac_cer"] for h in history], "x--", color="tab:olive", label=label("diacritised CER"))
    left.plot(x, [h["english_cer"] for h in history], "o-", color="tab:blue", label=label("English CER"))
    left.plot(x, [h["handwriting_cer"] for h in history], "o:", color="tab:purple", label=label("IAM handwriting CER"))
    left.plot(x, [h.get("khatt_cer") for h in history], "s:", color="tab:orange", label=label("KHATT CER"))
    left.plot(x, [h.get("muharaf_cer") for h in history], "s:", color="tab:brown", label=label("Muharaf CER"))
    right.plot(x, [h["eval_loss"] for h in history], "o-", color="tab:red", label=label("eval loss"))
    right.plot(x, [h["no_image_loss"] for h in history], "--", color="gray", label=label("blank-image loss"))
    for axis in (left, right):
        axis.axvline(offset, color="black", alpha=0.15)
        axis.text(offset, 1.0, f" stage {s}", transform=axis.get_xaxis_transform(), va="top", fontsize=9)
    offset = x[-1]
left.set_ylim(0, 1.5), left.set_xlabel("training examples seen"), left.set_ylabel("error rate"), left.legend(fontsize=8)
right.set_xlabel("training examples seen"), right.set_ylabel("loss"), right.legend()
fig.tight_layout()
fig.savefig(os.path.join(BASE_DIR, f"{RUN_NAME}_training_timeline.png"), dpi=150)
plt.show()

---
# Results 2: is it more accurate, and is it faster?

The final model is compared with **Qwen2.5-VL-3B-Instruct used as it is**, the model its vision encoder came from,
on the same evaluation images, prompts and GPU, both in half precision. For each model:

- **accuracy:** CER and WER on each evaluation set, and form field accuracy;
- **speed:** seconds per image and tokens per second, one image at a time (how a service answering single
  requests would run), after a warm-up;
- **size:** parameters and peak GPU memory.

The token limit comes from the image size for both models (v4 cut lines at 64 tokens here, which truncated the
diacritised set for both models and made that row meaningless). Keep the comparison honest when you write it up: the
3B model was not trained on these forms or this JSON format, so on forms it is at a disadvantage. The fair claims are
about speed and memory at a given accuracy, and about reading Arabic lines.

In [ ]:
@torch.no_grad()
def benchmark(path, label):
    free_memory()
    dtype = AMP_DTYPE if DEVICE.type == "cuda" else torch.float32
    model, processor = load_model(path, dtype=dtype)
    model.eval()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    read_images(model, processor, EVAL_SETS["arabic"][:2], batch_size=1)          # warm-up
    row, timings = {"model": label, "parameters_b": count_parameters(model) / 1e9}, []
    for name, examples in EVAL_SETS.items():
        predictions = read_images(model, processor, examples, batch_size=1, timings=timings)
        for metric, value in text_scores(predictions, [a for _, _, a in examples]).items():
            row[f"{name}_{metric}"] = value
    form_timings = []
    predictions = read_images(model, processor, EVAL_FORMS, kind="form", batch_size=1, timings=form_timings)
    row.update(form_scores(predictions, EVAL_FORMS))
    row["seconds_per_line"] = float(np.median([t for t, _ in timings]))
    row["seconds_per_form"] = float(np.median([t for t, _ in form_timings]))
    row["tokens_per_second"] = sum(n for _, n in timings + form_timings) / sum(t for t, _ in timings + form_timings)
    row["peak_gpu_gb"] = torch.cuda.max_memory_allocated() / 2**30 if DEVICE.type == "cuda" else 0.0
    del model
    free_memory()
    return row


rows = [benchmark(BASELINE_ID, "Qwen2.5-VL-3B (as is)"), benchmark(CHECKPOINTS[4], f"ours {RUN_NAME}, after stage 4")]

measures = [("parameters_b", "parameters (B)"), ("arabic_cer", "Arabic CER"), ("arabic_wer", "Arabic WER"),
            ("arabic_diac_cer", "diacritised CER"), ("bilingual_cer", "bilingual CER"), ("khatt_cer", "KHATT CER"), ("muharaf_cer", "Muharaf CER"),
            ("english_cer", "English CER"), ("handwriting_cer", "IAM CER"),
            ("field_accuracy", "form field accuracy"), ("valid_json", "valid JSON"),
            ("seconds_per_line", "seconds per line"), ("seconds_per_form", "seconds per form"),
            ("tokens_per_second", "tokens per second"), ("peak_gpu_gb", "peak GPU memory (GB)")]
print(f"{'':24s}" + "".join(f"{r['model']:>26s}" for r in rows))
for key, label in measures:
    print(f"{label:24s}" + "".join(f"{r[key]:26.3f}" for r in rows))
path = save_results(benchmark={"gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu", "rows": rows,
                               "max_pixels": MAX_PIXELS, "token_limits": TOKEN_LIMITS})
print(f"\nsaved to {path}")

---
# Results 3: public benchmarks other models were tested on

Our own evaluation sets come from the same generator as the training data, so they are easy for this model. The
sets below were made by other people and the model never trained on them:

| Set | What it is | Published results |
| --- | --- | --- |
| [KITAB-Bench](https://github.com/mbzuai-oryx/KITAB-Bench) (MBZUAI, 2025), 9 subsets | printed lines (PATS), synthetic lines with diacritics (SythenAR), slide text (ISI-PPT), scanned handwriting (KHATT), tablet handwriting (OnlineKhatt), words in street photos (EvArEST), **new:** historical handwriting (HistoryAr), handwritten words (ADAB), manuscript lines (Muharaf) | GPT-4o, GPT-4o-mini, Gemini-2.0-Flash, Qwen2-VL (paper, arXiv 2502.14949) |
| KHATT, test split | real Arabic handwriting, writers not seen in training | Arabic handwriting models |
| Muharaf, test split | historical Arabic handwriting | the Muharaf paper (NeurIPS 2024) |
| IAM handwriting, test split | real English handwriting | many handwriting models, e.g. TrOCR |

Four models are scored with the same prompt, token limits and scoring: **v5 after stage 4, v5 after stage 3** (the
pure-OCR checkpoint), **v4** (if its checkpoint is still on Drive) and **Qwen2.5-VL-3B as it is**. Three scores per
set: CER/WER as written, without diacritics, and Arabic-normalised, each with a 95% bootstrap interval on the raw CER.

**Measuring the leak.** v5 trains on KHATT and Muharaf lines that may share text with benchmark lines (part 1b kept
them and flagged them). For KHATT, OnlineKhatt, the KHATT test split and the two Muharaf sets, the lines are also
scored in two buckets: lines that share 4 words in a row with any training line of the same source, and lines that
do not. If the two buckets score alike, the overlap did not inflate the result; if the "clean" bucket is clearly
worse, that is the number to quote. v4 could not measure this because it had removed the overlapping lines.

`PUBLIC_N = 500` reads up to 500 images per set (about 20 minutes per model on an L4). Set `PUBLIC_N = None` to use
whole subsets, the same sizes as the published results. Every prediction is saved to `results_v5.json`.

In [ ]:
KITAB_PUBLISHED = {   # (CER, WER) on the whole subsets, from the KITAB-Bench project page (mbzuai-oryx.github.io/KITAB-Bench).
    "PATS":        {"GPT-4o": (0.23, 0.30), "GPT-4o-mini": (0.53, 0.71), "Gemini-2.0-Flash": (0.01, 0.02), "Qwen2-VL": (1.02, 1.02)},
    "SythenAR":    {"GPT-4o": (0.09, 0.20), "GPT-4o-mini": (0.14, 0.32), "Gemini-2.0-Flash": (0.07, 0.17), "Qwen2-VL": (0.59, 1.13)},
    "ISI-PPT":     {"GPT-4o": (0.08, 0.18), "GPT-4o-mini": (0.15, 0.31), "Gemini-2.0-Flash": (0.06, 0.15), "Qwen2-VL": (1.03, 1.06)},
    "KHATT":       {"GPT-4o": (0.45, 0.74), "GPT-4o-mini": (0.64, 0.91), "Gemini-2.0-Flash": (0.19, 0.45), "Qwen2-VL": (1.12, 5.04)},
    "OnlineKhatt": {"GPT-4o": (0.29, 0.63), "GPT-4o-mini": (0.41, 0.76), "Gemini-2.0-Flash": (0.17, 0.44), "Qwen2-VL": (1.30, 2.01)},
    "EvArEST":     {"GPT-4o": (0.20, 0.38), "GPT-4o-mini": (0.25, 0.51), "Gemini-2.0-Flash": (0.18, 0.36), "Qwen2-VL": (0.41, 0.95)},
}   # check the model order against the paper before quoting these; the new subsets' published numbers are in the paper's tables
KITAB_LINE_SUBSETS = ["PATS", "SythenAR", "ISI-PPT", "KHATT", "OnlineKhatt", "EvArEST", "HistoryAr", "ADAB", "Muharaf"]


def ngrams(text, n):
    words = normalise_arabic(re.sub(r"[^\w\s]", " ", text)).split()
    return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}


def take(ds, n):
    return [(ds[i]["image"].convert("RGB"), OCR_INSTRUCTION, ds[i]["text"]) for i in range(len(ds) if n is None else min(n, len(ds)))]


PUBLIC_SETS, LEAK_SOURCE = {}, {}
for name in KITAB_LINE_SUBSETS:
    ds = read_images_set(f"kitab_{name}")
    if ds is not None:
        PUBLIC_SETS[name] = take(ds, PUBLIC_N)
for label, ds in [("KHATT test", KHATT["test"]), ("Muharaf test", MUHARAF["test"]), ("IAM test", IAM["test"])]:
    PUBLIC_SETS[label] = take(ds, PUBLIC_N)
for name in ["KHATT", "OnlineKhatt", "KHATT test"]:
    LEAK_SOURCE[name] = "khatt"
for name in ["Muharaf", "Muharaf test"]:
    LEAK_SOURCE[name] = "muharaf"
TRAIN_4GRAMS = {"khatt": set().union(*(ngrams(t, 4) for t in KHATT["train"]["text"])),
                "muharaf": set().union(*(ngrams(t, 4) for t in MUHARAF["train"]["text"]))}
print("public sets:", ", ".join(f"{name} {len(examples)}" for name, examples in PUBLIC_SETS.items()))


@torch.no_grad()
def score_public(path, label):
    free_memory()
    model, processor = load_model(path, dtype=AMP_DTYPE if DEVICE.type == "cuda" else torch.float32)
    row = {"model": label}
    for name, examples in PUBLIC_SETS.items():
        start = time.time()
        predictions = read_images(model, processor, examples)
        truths = [answer for _, _, answer in examples]
        scores = text_scores(predictions, truths, interval=True)
        scores["images"] = len(examples)
        scores["predictions"] = [{"truth": t, "model": p} for p, t in zip(predictions, truths)]
        if name in LEAK_SOURCE:
            grams = TRAIN_4GRAMS[LEAK_SOURCE[name]]
            overlap = [not ngrams(t, 4).isdisjoint(grams) for t in truths]
            for bucket, flag in [("overlap", True), ("clean", False)]:
                picked = [i for i, o in enumerate(overlap) if o == flag]
                part = text_scores([predictions[i] for i in picked], [truths[i] for i in picked]) if picked else {"cer": float("nan"), "wer": float("nan")}
                scores[f"{bucket}_images"], scores[f"{bucket}_cer"], scores[f"{bucket}_wer"] = len(picked), part["cer"], part["wer"]
        row[name] = scores
        lo, hi = scores["cer_ci"]
        leak = f"  overlap {scores['overlap_images']} imgs CER {scores['overlap_cer']:.3f} / clean {scores['clean_images']} imgs CER {scores['clean_cer']:.3f}" if name in LEAK_SOURCE else ""
        print(f"  {label:22s} {name:13s} CER {scores['cer']:.3f} [{lo:.3f}, {hi:.3f}]  WER {scores['wer']:.3f}  "
              f"(no marks {scores['cer_no_marks']:.3f}, normalised {scores['cer_norm']:.3f})  length {scores['length']:.2f}  "
              f"{(time.time() - start) / 60:.1f} min{leak}")
        print(f"  {'':22s} {'':13s} truth: {truths[0]}\n  {'':22s} {'':13s} model: {predictions[0]}")
    del model
    free_memory()
    return row


public_models = [(f"ours {RUN_NAME} (stage 4)", CHECKPOINTS[4]), (f"ours {RUN_NAME} (stage 3)", CHECKPOINTS[3])]
if os.path.isdir(PREVIOUS_FINAL) and PREVIOUS_FINAL not in (CHECKPOINTS[3], CHECKPOINTS[4]):
    public_models.append((f"ours {PREVIOUS_RUN}", PREVIOUS_FINAL))
public_models.append(("Qwen2.5-VL-3B", BASELINE_ID))
public_rows = [score_public(path, label) for label, path in public_models]

In [ ]:
cell = lambda r, name, k="cer", w="wer": f"{r[name][k]:.2f} / {r[name][w]:.2f}" if name in r else ""
print(f"CER / WER (0 = perfect) as written. Up to {PUBLIC_N or 'all'} images per set; 95% intervals on CER in the next table.\n")
print(f"{'set':13s}" + "".join(f"{r['model']:>22s}" for r in public_rows) + "".join(f"{m + ' (pub.)':>20s}" for m in ["GPT-4o", "Gemini-2.0-Flash"]))
for name in PUBLIC_SETS:
    reference = KITAB_PUBLISHED.get(name, {})
    print(f"{name:13s}" + "".join(f"{cell(r, name):>22s}" for r in public_rows)
          + "".join(f"{(f'{reference[m][0]:.2f} / {reference[m][1]:.2f}' if m in reference else ''):>20s}" for m in ["GPT-4o", "Gemini-2.0-Flash"]))
print("\nCER with 95% bootstrap interval:")
for name in PUBLIC_SETS:
    print(f"{name:13s}" + "".join(f"{r[name]['cer']:.3f} [{r[name]['cer_ci'][0]:.3f}-{r[name]['cer_ci'][1]:.3f}]".rjust(22) for r in public_rows))
print("\nwithout diacritics (CER / WER):")
for name in PUBLIC_SETS:
    print(f"{name:13s}" + "".join(cell(r, name, "cer_no_marks", "wer_no_marks").rjust(22) for r in public_rows))
print("\nArabic-normalised (CER / WER):")
for name in PUBLIC_SETS:
    print(f"{name:13s}" + "".join(cell(r, name, "cer_norm", "wer_norm").rjust(22) for r in public_rows))
print("\nleak check, CER on lines that share 4 words with training handwriting vs lines that do not (images in brackets):")
for name in LEAK_SOURCE:
    if name in PUBLIC_SETS:
        print(f"{name:13s}" + "".join(f"{r[name]['overlap_cer']:.2f} ({r[name]['overlap_images']}) / {r[name]['clean_cer']:.2f} ({r[name]['clean_images']})".rjust(22) for r in public_rows))
path = save_results(public_benchmarks={"images_per_set": PUBLIC_N, "rows": public_rows, "kitab_published": KITAB_PUBLISHED})
print(f"\nsaved to {path} (with every prediction per set and model)")

---
# Results 4: decoding and resolution ablations

Two cheap questions about the final model, on the first 100 images of the handwriting sets and two printed sets:

- **Decoding.** v4's KHATT output sometimes fell into a loop ("20550 2056 2057 ..."). Greedy decoding is compared with
  `no_repeat_ngram_size=4` (a 4-gram may not repeat), `repetition_penalty=1.1`, and beam search with 3 beams. Repetition
  control can also damage text that legitimately repeats, which is why it is measured rather than switched on; if a
  setting wins, put it in `DECODING` in section A and the whole evaluation uses it for every model.
- **Resolution.** Training and evaluation allow up to 1,024 visual tokens (v4: 640). At inference the limit can be
  raised further; this compares 640, 1,024 and 1,280 on the printed sets, where small text is most likely to be lost.

In [ ]:
ABLATION_SETS = {name: PUBLIC_SETS[name][:100] for name in ["KHATT", "OnlineKhatt", "KHATT test", "IAM test", "PATS", "ISI-PPT"] if name in PUBLIC_SETS}
free_memory()
model, processor = load_model(CHECKPOINTS[4], dtype=AMP_DTYPE if DEVICE.type == "cuda" else torch.float32)
decodings = {"greedy": {}, "no_repeat_ngram_size=4": {"no_repeat_ngram_size": 4}, "repetition_penalty=1.1": {"repetition_penalty": 1.1},
             "num_beams=3": {"num_beams": 3}}
ablation = {"decoding": {}, "resolution": {}}
print(f"{'decoding':26s}" + "".join(f"{name:>14s}" for name in ABLATION_SETS))
for label, settings in decodings.items():
    line = {}
    for name, examples in ABLATION_SETS.items():
        predictions = read_images(model, processor, examples, decoding=settings, batch_size=4 if "num_beams" in settings else 8)
        line[name] = text_scores(predictions, [a for _, _, a in examples])
    ablation["decoding"][label] = line
    print(f"{label:26s}" + "".join(f"{line[name]['cer']:14.3f}" for name in ABLATION_SETS))
del model
free_memory()

print(f"\n{'max visual tokens':26s}" + "".join(f"{name:>14s}" for name in ["PATS", "ISI-PPT", "KHATT"] if name in ABLATION_SETS))
for tokens in [640, 1024, 1280]:
    model, processor = load_model(CHECKPOINTS[4], dtype=AMP_DTYPE if DEVICE.type == "cuda" else torch.float32, max_pixels=tokens * 28 * 28)
    line = {}
    for name in ["PATS", "ISI-PPT", "KHATT"]:
        if name in ABLATION_SETS:
            predictions = read_images(model, processor, ABLATION_SETS[name])
            line[name] = text_scores(predictions, [a for _, _, a in ABLATION_SETS[name]])
    ablation["resolution"][str(tokens)] = line
    print(f"{tokens:<26d}" + "".join(f"{line[name]['cer']:14.3f}" for name in line))
    del model
    free_memory()
save_results(ablations=ablation)

---
# Results 5: your own documents
Scored only if part 1b found `real_docs/labels.csv`. Images with a `text` label are read with the OCR instruction and
scored with CER/WER; images with a `json` label are read with the form instruction built from that JSON's keys and scored
by field accuracy. These are the only numbers in this notebook that say anything about real documents.

In [ ]:
if REAL_DOCS is None:
    print("no real documents prepared (optional): see part 1b v5, section 6")
else:
    model, processor = load_model(CHECKPOINTS[4], dtype=AMP_DTYPE if DEVICE.type == "cuda" else torch.float32, max_pixels=MAX_PIXELS_INFER)
    text_examples = [(r["image"].convert("RGB"), OCR_INSTRUCTION, r["text"]) for r in REAL_DOCS if r["text"]]
    json_examples = [(r["image"].convert("RGB"), form_instruction(list(json.loads(r["json"]))), r["json"]) for r in REAL_DOCS if r["json"]]
    real = {}
    if text_examples:
        predictions = read_images(model, processor, text_examples, kind="page", batch_size=2)
        real["text"] = {**text_scores(predictions, [t for _, _, t in text_examples], interval=True), "images": len(text_examples),
                        "predictions": [{"truth": t, "model": p} for p, (_, _, t) in zip(predictions, text_examples)]}
        print(f"OCR on {len(text_examples)} real images: CER {real['text']['cer']:.3f} [{real['text']['cer_ci'][0]:.3f}, {real['text']['cer_ci'][1]:.3f}], "
              f"WER {real['text']['wer']:.3f}, normalised CER {real['text']['cer_norm']:.3f}")
        for p, (_, _, t) in list(zip(predictions, text_examples))[:3]:
            print(f"   truth: {t[:120]!r}\n   model: {p[:120]!r}")
    if json_examples:
        predictions = read_images(model, processor, json_examples, kind="form", batch_size=2)
        real["json"] = {**form_scores(predictions, json_examples), "images": len(json_examples),
                        "predictions": [{"truth": t, "model": p} for p, (_, _, t) in zip(predictions, json_examples)]}
        print(f"fields on {len(json_examples)} real forms: field accuracy {real['json']['field_accuracy']:.3f}, valid JSON {real['json']['valid_json']:.3f}")
    save_results(real_documents=real)
    del model
    free_memory()

---
# Try the final model on your own image
Upload an image in Colab's Files panel and set its name below.

In [ ]:
MY_IMAGE = "my_form.png"           # change to your file name
if os.path.exists(MY_IMAGE):
    model, processor = load_model(4, max_pixels=MAX_PIXELS_INFER)
    img = Image.open(MY_IMAGE).convert("RGB")
    print("OCR:", read_images(model, processor, [(img, OCR_INSTRUCTION, "")], kind="page")[0])
    print("JSON:", read_images(model, processor, [(img, form_instruction(CORE_FIELDS), "")], kind="form")[0])
else:
    print(f"Upload an image named {MY_IMAGE} to try the model (or change MY_IMAGE).")

## Notes

- **Reading the progress checks.** First look at *eval loss* against *no-image loss*: the gap says whether the image
  is being used. Then *length*: far above 1 means the model does not stop; far below 1 means it stops early. CER and
  WER only make sense once both look reasonable.
- **Which switches to turn on first.** With the defaults (decoder as shipped, v3 stages 1–2 reused) the run costs about
  the same as v4 and tests the data changes, the full vision blocks, the token limits and the new scoring. Turn on
  `PRETRAIN_DECODER` for a second run if PATS/ISI-PPT errors are still plausible wrong words: it adds about 8 hours.
- **Colab disconnected mid-stage:** reopen the notebook, run sections A–F, then run the interrupted stage's cells
  from the top of that stage. It continues from the last progress check (`v5_stageN_in_progress.pt` on Drive).
- **Starting a stage again from scratch:** delete `v5_stageN_in_progress.pt` and `v5_stageN_start_scores.json` for that
  stage from `MyDrive/arabic_vlm`, then run its cells.
- **Out of memory:** in that stage's `train(...)` call, halve `micro_batch` and double `accum` (same examples per
  step), or lower `MAX_PIXELS` in section A (1,024 visual tokens is the v5 default; v4 used 640), or set
  `VISION_FULL_BLOCKS = 0`.
- **Drive space:** checkpoints are about 4.6 GB each (five with stage 0b, plus the assembled model), plus up to about
  1 GB of resume files while a stage runs. On a 15 GB Drive set `KEEP_ALL_CHECKPOINTS = False` in section A.
- **The data:** drawn Wikipedia and diacritised sentences, KHATT, Muharaf and IAM handwriting, SynthDoG pages and
  made-up forms, not real UAE government documents. Results on real scans will be lower until real examples are
  added; section Results 5 is where that is measured.
- **Licences:** the vision encoder is under the Qwen Research License; KHATT is for research use; Muharaf is
  CC BY-NC-SA 4.0; IAM is non-commercial research. A model trained on them is for research until those are cleared.